<a href="https://colab.research.google.com/github/gabriel-tfg/TFM_RAG_Pipeline/blob/main/PruebasTFM9_Llama_Libre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Llama de Meta es un modelo gated.
# Ejecuta esta celda si aparece un error 403 / gated repo.
# Antes debes aceptar la licencia del modelo en Hugging Face:
# https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct

from huggingface_hub import login
login()


In [ ]:
%pip install -q transformers


In [ ]:
%pip install -q faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 102.9 MB/s eta 0:00:00


1) Query configuration

In [ ]:
import requests
import xml.etree.ElementTree as ET
import json
import time
import re
from collections import defaultdict

QUERY_1 = """
"meta-analysis"[Publication Type] AND ("Menopause"[MeSH]) AND
("Quality of Life"[MeSH]
OR "Sleep Wake Disorders"[MeSH]
OR "Stress, Psychological"[MeSH]
OR "Fatigue"[MeSH]
OR "Anxiety"[MeSH]
OR "Cognition Disorders"[MeSH]
OR "Cognitive Dysfunction"[MeSH]
OR "Executive Function"[MeSH]
OR "Memory"[MeSH]
OR cognit*[tiab]
OR fatigue[tiab])
AND ((y_5[Filter]) AND (humans[Filter]) AND (female[Filter]) AND (middleagedaged[Filter]))
""".strip()

QUERY_2 = """
("meta-analysis"[Publication Type] AND "menopause/psychology"[MeSH Terms])
AND ((y_5[Filter]) AND (humans[Filter]) AND (female[Filter]) AND (middleagedaged[Filter]))
""".strip()

QUERY_3 = """
("meta-analysis"[Publication Type] AND "menopause/physiology"[MeSH Terms])
AND ((y_5[Filter]) AND (humans[Filter]) AND (female[Filter]) AND (middleagedaged[Filter]))
""".strip()

QUERY_4 = """
(
    review[Publication Type]
    AND menopause[Title/Abstract]
    AND (
        diagnosis[Title/Abstract]
        OR symptoms[Title/Abstract]
        OR perimenopause[Title/Abstract]
        OR transition[Title/Abstract]
    )
)
AND ((y_5[Filter]) AND (humans[Filter]) AND (female[Filter]))
""".strip()

QUERY_5 = """
(
    review[Publication Type]
    AND menopause[Title/Abstract]
    AND (
        diagnosis[Title/Abstract]
        OR amenorrhea[Title/Abstract]
        OR "menstrual irregularity"[Title/Abstract]
        OR perimenopause[Title/Abstract]
        OR "menopausal transition"[Title/Abstract]
        OR "clinical diagnosis"[Title/Abstract]
    )
)
AND ((humans[Filter]) AND (female[Filter]))
""".strip()

QUERY_6 = """
(
    (
        guideline[Title/Abstract]
        OR guidelines[Title/Abstract]
        OR "clinical practice guideline"[Title/Abstract]
        OR management[Title/Abstract]
        OR screening[Title/Abstract]
    )
    AND menopause[Title/Abstract]
)
AND ((humans[Filter]) AND (female[Filter]))
""".strip()

ALL_QUERIES = {
    "QUERY_1": QUERY_1,
    "QUERY_2": QUERY_2,
    "QUERY_3": QUERY_3,
    "QUERY_4": QUERY_4,
    "QUERY_5": QUERY_5,
    "QUERY_6": QUERY_6
}

RETMAX = 100


2) Pubmed search helpers

In [ ]:
ESEARCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
EFETCH_URL  = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
ELINK_URL   = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/elink.fcgi"

def pubmed_search(query, retmax=20):
    params = {
        "db": "pubmed",
        "term": query,
        "retmax": retmax,
        "retmode": "json"
    }

    r = requests.get(ESEARCH_URL, params=params, timeout=60)
    data = r.json()

    if "esearchresult" not in data:
        print("Error en PubMed ESearch:")
        print(data)
        return []

    return data["esearchresult"].get("idlist", [])

def pubmed_fetch_xml(pmids):
    if not pmids:
        return None

    params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml"
    }

    r = requests.get(EFETCH_URL, params=params, timeout=60)

    if not r.content.strip().startswith(b"<"):
        print("Respuesta no XML:")
        print(r.text[:500])
        return None

    return ET.fromstring(r.content)


def pubmed_fetch_xml_batched(pmids, batch_size=50):
    """
    Descarga artículos de PubMed por lotes para evitar errores XML
    cuando hay demasiados PMIDs.
    """
    all_articles = ET.Element("PubmedArticleSet")

    for i in range(0, len(pmids), batch_size):
        batch = pmids[i:i + batch_size]

        print(f"Fetching batch {i//batch_size + 1}: {len(batch)} PMIDs")

        root = pubmed_fetch_xml(batch)

        if root is None:
            continue

        for article in root.findall(".//PubmedArticle"):
            all_articles.append(article)

        time.sleep(1.0)

    return all_articles

def parse_pubmed_articles(root):
    docs = []
    if root is None:
        return docs

    for i, article in enumerate(root.findall(".//PubmedArticle")):
        pmid_elem = article.find(".//PMID")
        title_elem = article.find(".//ArticleTitle")
        abstract_elems = article.findall(".//Abstract/AbstractText")

        pmid = pmid_elem.text.strip() if pmid_elem is not None and pmid_elem.text else None
        if title_elem is None:
            continue

        title = "".join(title_elem.itertext()).strip()
        abstract = " ".join("".join(a.itertext()).strip() for a in abstract_elems) if abstract_elems else ""

        text = f"{title}\n\n{abstract}".strip()

        docs.append({
            "id": f"meta_{pmid}" if pmid else f"meta_{i}",
            "pmid": pmid,
            "type": "meta-analysis",
            "title": title,
            "text": text,
            "source_queries": []
        })

    return docs

# Buscar documentos combinando todas las queries definidas

ALL_QUERIES = {
    "QUERY_1": QUERY_1,
    "QUERY_2": QUERY_2,
    "QUERY_3": QUERY_3,
    "QUERY_4": QUERY_4,
    "QUERY_5": QUERY_5,
    "QUERY_6": QUERY_6
}

all_meta_pmids = set()
pmid_to_queries = defaultdict(list)

for q_name, q_text in ALL_QUERIES.items():
    pmids = pubmed_search(q_text, retmax=RETMAX)

    print(f"{q_name}: {len(pmids)} PMIDs")

    for p in pmids:
        all_meta_pmids.add(p)
        pmid_to_queries[p].append(q_name)

meta_pmids = sorted(all_meta_pmids)

print("\nTOTAL UNIQUE META PMIDs:", len(meta_pmids))

meta_root = pubmed_fetch_xml_batched(meta_pmids, batch_size=50)
meta_docs = parse_pubmed_articles(meta_root)

for d in meta_docs:
    pmid = d.get("pmid")
    d["source_queries"] = pmid_to_queries.get(pmid, [])

print("Meta-analysis docs:", len(meta_docs))
print(meta_docs[0]["title"] if meta_docs else "No docs")


QUERY_1: 34 PMIDs
QUERY_2: 21 PMIDs
QUERY_3: 57 PMIDs
QUERY_4: 100 PMIDs
QUERY_5: 100 PMIDs
QUERY_6: 100 PMIDs

TOTAL UNIQUE META PMIDs: 306
Fetching batch 1: 50 PMIDs
Fetching batch 2: 50 PMIDs
Fetching batch 3: 50 PMIDs
Fetching batch 4: 50 PMIDs
Fetching batch 5: 50 PMIDs
Fetching batch 6: 50 PMIDs
Fetching batch 7: 6 PMIDs
Meta-analysis docs: 306
The effect of resistance training programs on lean body mass in postmenopausal and elderly women: a meta-analysis of observational studies.


3) Link PubMed -> PMC

In [ ]:
import requests
import time

ID_CONVERTER_URL = "https://pmc.ncbi.nlm.nih.gov/tools/idconv/api/v1/articles/"

def get_pmcids_for_pmids(
    pmids,
    batch_size=100,
    tool="rag_pipeline",
    email="tu_email@ejemplo.com",
    max_retries=5,
    sleep_between_batches=1.0
):
    """
    Convierte PMIDs a PMCIDs usando el ID Converter de PMC por lotes.

    Devuelve un diccionario:
        {
            "PMID": "PMCID" o None
        }

    None puede significar:
    - el artículo no tiene PMCID asociado;
    - no está disponible en PubMed Central;
    - el conversor no devolvió PMCID para ese PMID.
    """

    pmids = [str(p) for p in pmids if p]
    pmids = list(dict.fromkeys(pmids))  # elimina duplicados manteniendo orden

    meta_to_pmc = {}

    for i in range(0, len(pmids), batch_size):
        batch = pmids[i:i + batch_size]
        batch_number = i // batch_size + 1

        params = {
            "ids": ",".join(batch),
            "format": "json",
            "tool": tool,
            "email": email
        }

        print(f"Convirtiendo batch {batch_number}: {len(batch)} PMIDs")

        data = None

        for attempt in range(max_retries):
            try:
                r = requests.get(
                    ID_CONVERTER_URL,
                    params=params,
                    timeout=30
                )

                if r.status_code == 429:
                    wait = 2 ** attempt
                    print(
                        f"[Rate limit] Batch {batch_number}: "
                        f"429 Too Many Requests. Reintentando en {wait}s..."
                    )
                    time.sleep(wait)
                    continue

                r.raise_for_status()

                content_type = r.headers.get("Content-Type", "")

                if "json" not in content_type.lower():
                    print(f"[Aviso] Batch {batch_number}: la respuesta no es JSON.")
                    print("Content-Type:", content_type)
                    print("Respuesta:", r.text[:300])
                    break

                if not r.text.strip():
                    print(f"[Aviso] Batch {batch_number}: respuesta vacía.")
                    break

                data = r.json()
                break

            except requests.exceptions.RequestException as e:
                wait = 2 ** attempt
                print(
                    f"[Error HTTP] Batch {batch_number}, intento {attempt + 1}: {e}. "
                    f"Reintentando en {wait}s..."
                )
                time.sleep(wait)

            except ValueError as e:
                print(f"[Error JSON] Batch {batch_number}: {e}")
                print("Respuesta recibida:", r.text[:300])
                break

        # Si el batch falló del todo, marcamos sus PMIDs como None
        if data is None:
            for pmid in batch:
                meta_to_pmc.setdefault(pmid, None)
            time.sleep(sleep_between_batches)
            continue

        records = data.get("records", [])

        # Inicializamos todos como None; luego rellenamos los que tengan PMCID
        for pmid in batch:
            meta_to_pmc.setdefault(pmid, None)

        for record in records:
            pmid = record.get("pmid")
            pmcid = record.get("pmcid")

            if pmid:
                meta_to_pmc[str(pmid)] = pmcid if pmcid else None

        time.sleep(sleep_between_batches)

    return meta_to_pmc


# =========================
# Conversión PMID -> PMCID
# =========================

meta_pmids_for_pmc = [
    d["pmid"]
    for d in meta_docs
    if d.get("pmid")
]

meta_to_pmc = get_pmcids_for_pmids(
    meta_pmids_for_pmc,
    batch_size=100,
    tool="rag_pipeline",
    email="tu_email@ejemplo.com",
    max_retries=5,
    sleep_between_batches=1.0
)

print(meta_to_pmc)

print("Total PMIDs:", len(meta_to_pmc))
print("Con PMCID:", sum(1 for v in meta_to_pmc.values() if v))
print("Sin PMCID:", sum(1 for v in meta_to_pmc.values() if not v))

Convirtiendo batch 1: 100 PMIDs
Convirtiendo batch 2: 100 PMIDs
Convirtiendo batch 3: 100 PMIDs
Convirtiendo batch 4: 6 PMIDs
{'33880736': 'PMC8595144', '34246664': None, '34444691': 'PMC8398438', '34455600': None, '34740274': None, '34758929': None, '34805742': 'PMC8598284', '36147572': 'PMC9486389', '36378126': None, '37477236': None, '37697662': 'PMC10594314', '37945913': None, '37960761': 'PMC10637479', '38328822': None, '38342411': None, '38353251': None, '38390196': 'PMC10882717', '38489855': None, '38553980': None, '38563867': None, '38583073': None, '38619017': None, '38642901': None, '38735578': None, '38744142': None, '38824654': None, '38876649': None, '38956480': 'PMC11220992', '38977980': 'PMC11229230', '39026541': 'PMC11257064', '39127644': 'PMC11316363', '39145901': None, '39214463': None, '39328990': 'PMC11425856', '39431435': 'PMC11503877', '39542236': None, '39609539': 'PMC11747320', '39799097': None, '39856668': 'PMC11762881', '39907867': None, '39919442': None, '399

4) PMC XML retrieval + reference extraction

In [ ]:
from collections import defaultdict
import requests
import xml.etree.ElementTree as ET
import time
import re

PMC_OAI_URL = "https://pmc.ncbi.nlm.nih.gov/api/oai/v1/mh/"

def fetch_pmc_xml_oai(pmcid):
    if not pmcid:
        return None

    pmc_num = pmcid.replace("PMC", "").strip()

    params = {
        "verb": "GetRecord",
        "identifier": f"oai:pubmedcentral.nih.gov:{pmc_num}",
        "metadataPrefix": "pmc"
    }

    try:
        r = requests.get(PMC_OAI_URL, params=params, timeout=30)
        r.raise_for_status()

        if not r.text.strip():
            print(f"[Aviso] {pmcid}: respuesta vacía")
            return None

        return r.text

    except requests.exceptions.RequestException as e:
        print(f"[Error HTTP] {pmcid}: {e}")
        return None

def extract_article_xml_from_oai(oai_xml_text):
    if not oai_xml_text:
        return None

    try:
        root = ET.fromstring(oai_xml_text)
    except Exception as e:
        print(f"[Error XML OAI] {e}")
        return None

    ns = {
        "oai": "http://www.openarchives.org/OAI/2.0/"
    }

    metadata = root.find(".//oai:metadata", ns)
    if metadata is None or len(metadata) == 0:
        print("[Aviso] No se encontró bloque <metadata> en la respuesta OAI")
        return None

    article_elem = list(metadata)[0]

    try:
        return ET.tostring(article_elem, encoding="unicode")
    except Exception as e:
        print(f"[Error serializando article XML] {e}")
        return None

def extract_references_from_pmc_xml(article_xml_text):
    refs = []
    if not article_xml_text:
        return refs

    try:
        root = ET.fromstring(article_xml_text)
    except Exception as e:
        print(f"[Error parseando article XML] {e}")
        return refs

    for i, ref in enumerate(root.findall(".//{*}ref")):
        ref_text = " ".join(t.strip() for t in ref.itertext() if t and t.strip())

        pmid = None
        doi = None
        year = None
        first_author = None

        ref_id = ref.attrib.get("id", f"ref_{i}")

        for pubid in ref.findall(".//{*}pub-id"):
            id_type = pubid.attrib.get("pub-id-type", "").lower()
            if pubid.text:
                if id_type == "pmid" and pmid is None:
                    pmid = pubid.text.strip()
                elif id_type == "doi" and doi is None:
                    doi = pubid.text.strip()

        year_match = re.search(r"\b(19|20)\d{2}\b", ref_text)
        if year_match:
            year = year_match.group(0)

        surname_elem = ref.find(".//{*}surname")
        if surname_elem is not None and surname_elem.text:
            first_author = surname_elem.text.strip().lower()

        refs.append({
            "ref_id": ref_id,
            "pmid": pmid,
            "doi": doi,
            "year": year,
            "first_author": first_author,
            "ref_text": ref_text
        })

    return refs


4.1) Included study detection (heuristic)

In [ ]:
import xml.etree.ElementTree as ET

def extract_candidate_sections(article_xml_text):
    sections = []
    if not article_xml_text:
        return sections

    try:
        root = ET.fromstring(article_xml_text)
    except Exception as e:
        print(f"[Error parseando XML para secciones] {e}")
        return sections

    for sec in root.findall(".//{*}sec"):
        title_elem = sec.find("./{*}title")
        title = ""
        if title_elem is not None:
            title = " ".join(t.strip() for t in title_elem.itertext() if t and t.strip()).lower()

        sec_text = " ".join(t.strip() for t in sec.itertext() if t and t.strip())
        sections.append({
            "title": title,
            "text": sec_text.lower()
        })

    return sections

def extract_candidate_tables(article_xml_text):
    tables = []
    if not article_xml_text:
        return tables

    try:
        root = ET.fromstring(article_xml_text)
    except Exception as e:
        print(f"[Error parseando XML para tablas] {e}")
        return tables

    for tbl in root.findall(".//{*}table-wrap"):
        label_elem = tbl.find(".//{*}label")
        caption_elem = tbl.find(".//{*}caption")

        label = " ".join(label_elem.itertext()).strip().lower() if label_elem is not None else ""
        caption = " ".join(caption_elem.itertext()).strip().lower() if caption_elem is not None else ""
        text = " ".join(t.strip() for t in tbl.itertext() if t and t.strip()).lower()

        tables.append({
            "label": label,
            "caption": caption,
            "text": text
        })

    return tables

def score_reference_as_included(ref, sections, tables):
    score = 0
    evidence = []

    first_author = ref.get("first_author")
    year = ref.get("year")

    if ref.get("pmid"):
        score += 2
        evidence.append("has_pmid")

    relevant_titles = [
        "included studies",
        "characteristics of included studies",
        "study characteristics",
        "included study",
        "results",
        "studies included",
        "study selection"
    ]

    for sec in sections:
        sec_title = sec["title"]
        sec_text = sec["text"]

        title_is_relevant = any(key in sec_title for key in relevant_titles)

        if first_author and year and first_author in sec_text and year in sec_text:
            if title_is_relevant:
                score += 5
                evidence.append(f"section:{sec_title}")
            else:
                score += 2
                evidence.append("author_year_in_section")

    table_keywords = [
        "included",
        "characteristics",
        "study",
        "studies"
    ]

    for tbl in tables:
        tbl_meta = f"{tbl['label']} {tbl['caption']}"
        tbl_text = tbl["text"]
        table_is_relevant = any(k in tbl_meta for k in table_keywords)

        if first_author and year and first_author in tbl_text and year in tbl_text:
            if table_is_relevant:
                score += 6
                evidence.append(f"table:{tbl_meta[:80]}")
            else:
                score += 3
                evidence.append("author_year_in_table")

    return score, evidence

def extract_included_study_candidates(article_xml_text, min_score=5):
    refs = extract_references_from_pmc_xml(article_xml_text)
    sections = extract_candidate_sections(article_xml_text)
    tables = extract_candidate_tables(article_xml_text)

    included = []
    all_scored = []

    for ref in refs:
        score, evidence = score_reference_as_included(ref, sections, tables)

        item = {
            **ref,
            "score": score,
            "evidence": evidence
        }
        all_scored.append(item)

        if score >= min_score:
            included.append(item)

    included = sorted(included, key=lambda x: x["score"], reverse=True)
    all_scored = sorted(all_scored, key=lambda x: x["score"], reverse=True)

    return included, all_scored


4.2) Test included-study detection + build mappings

In [ ]:
from collections import defaultdict
import time

test_pmcid = "PMC12835450"

oai_xml = fetch_pmc_xml_oai(test_pmcid)
print(oai_xml[:500] if oai_xml else "No OAI XML")

article_xml = extract_article_xml_from_oai(oai_xml)
print(article_xml[:500] if article_xml else "No article XML")

refs = extract_references_from_pmc_xml(article_xml)
print("N refs:", len(refs))
print("First refs:", refs[:3])

included, all_scored = extract_included_study_candidates(article_xml, min_score=5)
print("Included candidates:", len(included))
print("Top included candidates:", included[:5])

meta_included_candidates = defaultdict(list)
meta_all_scored_refs = defaultdict(list)

for d in meta_docs:
    pmid = d.get("pmid")
    pmcid = meta_to_pmc.get(pmid)

    if pmcid:
        oai_xml = fetch_pmc_xml_oai(pmcid)
        article_xml = extract_article_xml_from_oai(oai_xml)

        included, all_scored = extract_included_study_candidates(article_xml, min_score=5)

        meta_included_candidates[pmid] = included
        meta_all_scored_refs[pmid] = all_scored

        print(f"Meta PMID {pmid}: included candidates = {len(included)} / all refs = {len(all_scored)}")
        time.sleep(0.34)

print("Example meta PMID:", next(iter(meta_included_candidates)) if meta_included_candidates else "No meta")





<OAI-PMH xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
         xsi:schemaLocation="http://www.openarchives.org/OAI/2.0/ http://www.openarchives.org/OAI/2.0/OAI-PMH.xsd">
    <responseDate>2026-06-12T12:09:36Z</responseDate>
    <request verb="GetRecord" identifier="oai:pubmedcentral.nih.gov:12835450" metadataPrefix="pmc">https://pmc.ncbi.nlm.nih.gov/api/oai/v1/mh/</request>
    
    <GetRecord>
        <record>
    <header>
    <identifier
<ns0:article xmlns:ns0="https://jats.nlm.nih.gov/ns/archiving/1.4/" xmlns:ns2="http://www.niso.org/schemas/ali/1.0/" xmlns:ns3="http://www.w3.org/1999/xlink" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="https://jats.nlm.nih.gov/ns/archiving/1.4/ https://jats.nlm.nih.gov/archiving/1.4/xsd/JATS-archivearticle1-4.xsd" xml:lang="en" article-type="research-article" dtd-version="1.4"><ns0:processing-meta base-tagset="archiving" mathml-version="3.0" table-model="xhtml" tag

5) Collect cited PMIDs from references

In [ ]:
included_pmids = set()

for meta_pmid, refs in meta_included_candidates.items():
    for ref in refs:
        if ref.get("pmid"):
            included_pmids.add(str(ref["pmid"]))

included_pmids = sorted(included_pmids)

print("Included-study candidate PMIDs found:", len(included_pmids))
print(included_pmids[:10])


Included-study candidate PMIDs found: 1063
['10183310', '10206619', '10411917', '10419000', '10687834', '10690445', '10718506', '10817075', '10941954', '11034522']


5.2) Save included PMIDs + prepare new PMIDs

In [ ]:
import json

# 1. Guardar todos los PMIDs incluidos (heurísticos)
with open("included_pmids.json", "w", encoding="utf-8") as f:
    json.dump(included_pmids, f, ensure_ascii=False, indent=2)

# 2. PMIDs semilla (metaanálisis)
seed_pmids = {
    str(d["pmid"])
    for d in meta_docs
    if d.get("pmid")
}

# 3. Filtrar: quitar los metaanálisis
new_included_pmids = [
    str(pmid) for pmid in included_pmids
    if str(pmid) not in seed_pmids
]

# 4. Ordenar
new_included_pmids = sorted(set(new_included_pmids))

# 5. Guardar
with open("new_included_pmids.json", "w", encoding="utf-8") as f:
    json.dump(new_included_pmids, f, ensure_ascii=False, indent=2)

print("Seed PMIDs:", len(seed_pmids))
print("All included PMIDs:", len(included_pmids))
print("New included PMIDs:", len(new_included_pmids))
print(new_included_pmids[:10])


Seed PMIDs: 306
All included PMIDs: 1063
New included PMIDs: 1062
['10183310', '10206619', '10411917', '10419000', '10687834', '10690445', '10718506', '10817075', '10941954', '11034522']


6) Fetch cited papers

In [ ]:
included_root = pubmed_fetch_xml(new_included_pmids[:200])
included_docs_raw = parse_pubmed_articles(included_root)

included_docs = []
for d in included_docs_raw:
    included_docs.append({
        "id": f"included_{d['pmid']}" if d.get("pmid") else d.get("id"),
        "pmid": d.get("pmid"),
        "type": "included-study-candidate",
        "title": d.get("title"),
        "text": d.get("text")
    })

print("Included-study candidate docs:", len(included_docs))
print(included_docs[0]["title"] if included_docs else "No docs")


Included-study candidate docs: 200
The MOS 36-item short form health survey. A conceptual analysis.


7) Final corpus + repository files

In [ ]:
import json

seen = set()
docs = []

for d in meta_docs + included_docs:
    pmid = d.get("pmid")
    if pmid and pmid not in seen:
        docs.append(d)
        seen.add(pmid)

print("TOTAL DOCS IN CORPUS:", len(docs))

# guardar corpus
with open("corpus_meta_plus_included.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False, indent=2)

# guardar mapping meta -> included candidates
with open("meta_included_map.json", "w", encoding="utf-8") as f:
    json.dump(dict(meta_included_candidates), f, ensure_ascii=False, indent=2)

print("Saved corpus_meta_plus_included.json")
print("Saved meta_included_map.json")


TOTAL DOCS IN CORPUS: 506
Saved corpus_meta_plus_included.json
Saved meta_included_map.json


8) Chunking

In [ ]:
import re

def chunk_text_sentences(text, chunk_chars=1200, overlap_chars=200):
    if not text or not text.strip():
        return []

    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks = []
    cur = ""

    for s in sents:
        s = s.strip()
        if not s:
            continue

        # si una sola frase ya supera el tamaño, la troceamos de forma defensiva
        if len(s) > chunk_chars:
            if cur:
                chunks.append(cur)
                cur = ""

            start = 0
            while start < len(s):
                end = start + chunk_chars
                piece = s[start:end]
                chunks.append(piece)
                start += max(1, chunk_chars - overlap_chars)
            continue

        if len(cur) + len(s) + 1 <= chunk_chars:
            cur = (cur + " " + s).strip()
        else:
            if cur:
                chunks.append(cur)
            cur = s

    if cur:
        chunks.append(cur)

    out = []
    for i, c in enumerate(chunks):
        if i == 0:
            out.append(c)
        else:
            prev = out[-1]
            overlap = prev[-overlap_chars:] if len(prev) > overlap_chars else prev
            out.append((overlap + " " + c).strip())

    return out


# reconstruir chunks con metadatos útiles para RAG
chunks = []

for d in docs:
    doc_text = d.get("text", "")
    doc_chunks = chunk_text_sentences(doc_text, chunk_chars=1200, overlap_chars=200)

    for i, c in enumerate(doc_chunks):
        chunks.append({
            "chunk_id": f"{d['id']}_chunk_{i}",
            "source_id": d.get("id"),
            "pmid": d.get("pmid"),
            "type": d.get("type"),
            "source_type_priority": 2 if d.get("type") == "meta-analysis" else 1,
            "title": d.get("title"),
            "chunk_index": i,
            "text": c
        })

print("N chunks:", len(chunks))
print("Chunk example:", chunks[0]["text"][:300] if chunks else "No chunks")


N chunks: 1007
Chunk example: The effect of resistance training programs on lean body mass in postmenopausal and elderly women: a meta-analysis of observational studies. Aging and menopause are associated with morphological and functional changes which may lead to loss of muscle mass and therefore quality of life. Resistance tra


9) Embeddings + FAISS + retrieve()

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Modelo de embeddings
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Textos de los chunks
texts = [c["text"] for c in chunks]

# Crear embeddings
emb = embed_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

emb = emb.astype("float32")

# Crear índice FAISS con similitud coseno vía inner product
dim = emb.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(emb)

print("Embeddings shape:", emb.shape)
print("FAISS index size:", index.ntotal)

def keyword_score(query, title, text):
    """
    Re-ranking ligero por palabras clave biomédicas.
    Premia coincidencias específicas entre query expandida y título/texto.
    """
    q = query.lower()
    title_l = (title or "").lower()
    text_l = (text or "").lower()

    score = 0.0

    keyword_groups = {
        "diagnosis": [
            "diagnosis", "clinical diagnosis", "menopausal transition",
            "perimenopause", "amenorrhea", "amenorrhoea",
            "menstrual irregularity", "irregular periods",
            "12 months", "final menstrual period"
        ],
        "vasomotor_mechanism": [
            "vasomotor symptoms", "hot flashes", "hot flushes",
            "night sweats", "thermoregulatory", "thermoregulation",
            "hypothalamic", "neurokinin", "estrogen decline"
        ],
        "vasomotor_treatment": [
            "hot flashes", "hot flushes", "vasomotor",
            "treatment", "therapy", "management", "fezolinetant",
            "elinzanetant", "hormone therapy", "nonhormonal"
        ],
        "cbt_insomnia": [
            "cbt-i", "cbti", "cognitive behavioral therapy", "cognitive behavioural therapy",
            "insomnia", "insomnia severity", "sleep quality", "sleep restriction",
            "sleep hygiene", "isi", "psqi"
        ],
        "mental_health": [
            "depression", "depressive", "anxiety", "stress", "psychological",
            "mood", "irritability", "mental health", "quality of life"
        ],
        "mindfulness": [
            "mindfulness", "mbsr", "mindfulness-based stress reduction",
            "psychoeducation", "mind-body"
        ],
        "cardio": [
            "cardiovascular", "heart disease", "cardiometabolic", "coronary",
            "lipid", "blood pressure", "metabolic"
        ],
        "hormone_therapy": [
            "hormone therapy", "menopausal hormone therapy", "mht", "hrt",
            "estrogen", "oestradiol", "estradiol"
        ]
    }

    for group_terms in keyword_groups.values():
        query_hits = sum(1 for term in group_terms if term in q)
        if query_hits > 0:
            title_hits = sum(1 for term in group_terms if term in title_l)
            text_hits = sum(1 for term in group_terms if term in text_l)

            score += 0.035 * title_hits
            score += 0.012 * min(text_hits, 5)

    specific_terms = [
        "clinical diagnosis", "menopausal transition", "amenorrhea",
        "menstrual irregularity", "cbt-i", "cbti",
        "cognitive behavioral therapy", "cognitive behavioural therapy",
        "hot flashes", "hot flushes", "vasomotor symptoms",
        "thermoregulatory", "mindfulness", "depression", "anxiety",
        "cardiovascular", "heart disease"
    ]

    for term in specific_terms:
        if term in q and term in title_l:
            score += 0.06

    return score


def noise_penalty(query, title, text):
    """
    Penaliza papers que aparecen mucho pero no son centrales para ciertas preguntas.
    """
    q = query.lower()
    title_l = (title or "").lower()
    text_l = (text or "").lower()
    full = title_l + " " + text_l

    penalty = 0.0

    if "massage" in full and "massage" not in q and "masaje" not in q:
        penalty += 0.10

    if any(term in full for term in ["questionnaire", "scale", "instrument", "validation"]):
        if any(term in q for term in [
            "treatment", "therapy", "manage", "improve", "mejor", "tratamiento",
            "qué puedo hacer", "sofocos", "insomnio", "mindfulness", "cbt"
        ]):
            penalty += 0.05

    if any(term in full for term in ["artificial intelligence", "digital health", "app"]):
        if not any(term in q for term in ["app", "ai", "ia", "artificial intelligence", "digital"]):
            penalty += 0.08

    diagnosis_query = any(term in q for term in [
        "menopause diagnosis", "clinical diagnosis", "menopausal transition",
        "amenorrhea", "amenorrhoea", "menstrual irregularity",
        "cómo sé", "como se", "saber si", "menopausia o no", "es menopausia"
    ])

    mental_query = any(term in q for term in [
        "mental", "depression", "depresión", "depresion", "anxiety", "ansiedad",
        "stress", "estrés", "estres", "mood", "ánimo", "animo", "irritable"
    ])

    if diagnosis_query and not mental_query:
        mental_focus_terms = [
            "mental health", "depression", "depressive", "anxiety",
            "psychological", "psychiatric", "cognitive", "mood"
        ]
        diagnosis_anchor_terms = [
            "menstrual", "amenorrhea", "amenorrhoea", "irregular periods",
            "menopausal transition", "final menstrual period", "diagnosis",
            "hot flashes", "hot flushes", "night sweats", "vasomotor",
            "vaginal dryness", "genitourinary"
        ]
        if any(term in title_l for term in mental_focus_terms):
            penalty += 0.12
        if any(term in full for term in mental_focus_terms) and not any(term in full for term in diagnosis_anchor_terms):
            penalty += 0.10

    return penalty


def retrieve(query, k=10, initial_k=80, max_per_doc=1,
             use_type_priority=True, use_keyword_rerank=True):
    """
    Retrieval FAISS + re-ranking biomédico ligero.
    Ajusta el ranking según intención probable para evitar contaminación temática.
    """

    if not query or not query.strip():
        return []

    if len(chunks) == 0:
        return []

    initial_k = min(initial_k, len(chunks))
    k = min(k, len(chunks))

    q_l = query.lower()

    diagnosis_query = any(t in q_l for t in [
        "menopause diagnosis", "clinical diagnosis", "menopausal transition",
        "amenorrhea", "amenorrhoea", "menstrual irregularity",
        "cómo sé", "como se", "saber si", "menopausia o no", "es menopausia"
    ])

    vasomotor_query = any(t in q_l for t in [
        "hot flash", "hot flashes", "hot flush", "hot flushes",
        "vasomotor", "sofoco", "sofocos", "calor", "sudor", "sudores"
    ])

    cardiovascular_risk_query = any(t in q_l for t in [
        "cardiovascular risk", "cardiometabolic risk", "coronary heart disease",
        "heart disease", "vascular health", "vascular aging",
        "enfermedad cardiovascular", "riesgo cardiovascular",
        "corazón", "corazon", "cardiovascular"
    ]) and not any(t in q_l for t in [
        "hormone therapy", "menopausal hormone therapy", "terapia hormonal",
        "tratamiento hormonal", "mht", "hrt", "prescribing", "prescripción",
        "prescripcion"
    ])

    cause_query = any(t in q_l for t in [
        "por qué", "por que", "a qué se debe", "a que se debe",
        "qué causa", "que causa", "why", "cause", "causes", "mecanismo"
    ])

    treatment_query_terms = [
        "qué puedo hacer", "que puedo hacer",
        "tratamiento", "tratar", "manejo",
        "opciones", "aliviar", "mejorar",
        "treatment", "therapy", "management",
        "intervention", "relief"
    ]

    treatment_query = any(t in q_l for t in treatment_query_terms)

    treatment_doc_terms = [
        "treatment", "therapy", "management",
        "intervention", "effective", "efficacy",
        "fezolinetant", "elinzanetant",
        "hormone therapy", "nonhormonal",
        "non-hormonal", "cbt", "cognitive behavioral",
        "mindfulness", "exercise", "acupuncture"
    ]

    diagnosis_doc_terms = [
        "clinical diagnosis", "diagnosis", "menopausal transition",
        "perimenopause", "amenorrhea", "amenorrhoea", "menstrual",
        "irregular periods", "final menstrual period", "straw",
        "clinical practice guideline", "clinical update",
        "evaluation and management", "menopause consultation"
    ]

    vasomotor_mechanism_terms = [
        "thermoregulatory", "thermoregulation", "hypothalamic",
        "neurokinin", "estrogen decline", "pathophysiology",
        "mechanism", "mechanisms", "vasomotor symptoms"
    ]

    cardiovascular_risk_terms = [
        "cardiovascular risk", "cardiometabolic risk", "coronary heart disease",
        "vascular health", "vascular aging", "atherosclerosis",
        "blood pressure", "lipid", "lipids", "metabolic syndrome",
        "estrogen decline", "menopause transition", "traditional risk factors",
        "nontraditional risk factors"
    ]

    hormone_prescribing_terms = [
        "eligibility criteria", "prescribing menopausal hormone therapy",
        "prescription of menopausal hormone therapy", "consensus document",
        "agreement document", "hormone therapy to patients",
        "contraindications", "indications"
    ]

    q = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, idxs = index.search(q, initial_k)

    candidates = []

    for score, i in zip(scores[0], idxs[0]):

        ch = chunks[i]

        title = ch.get("title", "")
        text = ch.get("text", "")
        full_text = (title + " " + text).lower()

        adjusted_score = float(score)

        if use_type_priority:
            adjusted_score += 0.02 * float(ch.get("source_type_priority", 1))

        source_queries = ch.get("source_queries", [])

        if "QUERY_4" in source_queries:
            adjusted_score += 0.04

        if "QUERY_5" in source_queries:
            adjusted_score += 0.10 if diagnosis_query else 0.06

        if "QUERY_6" in source_queries:
            adjusted_score += 0.08 if treatment_query else 0.04

        k_score = 0.0
        n_penalty = 0.0

        if use_keyword_rerank:
            k_score = keyword_score(query, title, text)
            n_penalty = noise_penalty(query, title, text)

            adjusted_score += k_score
            adjusted_score -= n_penalty

        if diagnosis_query:
            if any(t in full_text for t in diagnosis_doc_terms):
                adjusted_score += 0.14
            if any(t in title.lower() for t in [
                "clinical practice guideline", "clinical update",
                "evaluation and management", "menopause and the perimenopause",
                "menopause consultation"
            ]):
                adjusted_score += 0.30
            if any(t in full_text for t in ["amenorrhea", "amenorrhoea", "menstrual", "final menstrual period", "irregular periods"]):
                adjusted_score += 0.08
            if any(t in full_text for t in ["vasomotor symptoms", "hot flashes", "hot flushes"]) and not any(t in full_text for t in [
                "amenorrhea", "amenorrhoea", "menstrual", "final menstrual period",
                "irregular periods", "clinical diagnosis", "diagnosis"
            ]):
                adjusted_score -= 0.16
            if any(t in full_text for t in ["mental health", "depression", "anxiety", "cognitive", "mood"]):
                if not any(t in full_text for t in [
                    "menstrual", "amenorrhea", "amenorrhoea", "hot flashes",
                    "hot flushes", "night sweats", "vasomotor", "vaginal dryness",
                    "genitourinary", "diagnosis"
                ]):
                    adjusted_score -= 0.14
            if any(t in title.lower() for t in ["aesthetic", "aesthetically", "skin", "hair", "alopecia"]):
                adjusted_score -= 0.35
            elif any(t in full_text for t in ["aesthetic", "aesthetically", "skin", "hair", "alopecia"]):
                if not any(t in full_text for t in ["amenorrhea", "amenorrhoea", "menstrual", "final menstrual period"]):
                    adjusted_score -= 0.22
            if any(t in title.lower() for t in [
                "resistance training", "physical activity", "exercise",
                "reducing hot flushes", "reducing hot flashes",
                "hot flushes", "hot flashes", "vasomotor symptoms"
            ]):
                if not any(t in full_text for t in [
                    "amenorrhea", "amenorrhoea", "menstrual",
                    "final menstrual period", "clinical diagnosis"
                ]):
                    adjusted_score -= 0.75
                if not any(t in title.lower() for t in [
                    "clinical", "guideline", "diagnosis", "menstrual",
                    "amenorrhea", "amenorrhoea", "final menstrual period",
                    "menopause and the perimenopause"
                ]):
                    adjusted_score -= 0.35
            if any(t in title.lower() for t in [
                "duration of menopausal vasomotor symptoms",
                "duration of vasomotor symptoms",
                "vasomotor symptoms over the menopause transition"
            ]):
                adjusted_score -= 0.35

        if cardiovascular_risk_query:
            if any(t in full_text for t in cardiovascular_risk_terms):
                adjusted_score += 0.14
            if any(t in title.lower() for t in ["cardiovascular health during menopause transition", "risk factors"]):
                adjusted_score += 0.08
            if any(t in full_text for t in hormone_prescribing_terms):
                adjusted_score -= 0.22
            elif any(t in full_text for t in ["hormone therapy", "menopausal hormone therapy", "mht", "hrt"]):
                adjusted_score -= 0.08

        if vasomotor_query and cause_query and not treatment_query:
            if any(t in full_text for t in vasomotor_mechanism_terms):
                adjusted_score += 0.09
            if any(t in full_text for t in treatment_doc_terms):
                adjusted_score -= 0.05

        if treatment_query:
            if any(t in full_text for t in treatment_doc_terms):
                adjusted_score += 0.08
            if any(t in full_text for t in [
                "prevalence", "mapping global prevalence",
                "relationship between severity", "quality of life"
            ]):
                adjusted_score -= 0.05

        candidates.append({
            "score": float(score),
            "keyword_score": float(k_score),
            "noise_penalty": float(n_penalty),
            "adjusted_score": float(adjusted_score),
            "chunk_id": ch.get("chunk_id"),
            "source_id": ch.get("source_id"),
            "pmid": ch.get("pmid"),
            "type": ch.get("type"),
            "title": title,
            "chunk_index": ch.get("chunk_index"),
            "text": text
        })

    candidates = sorted(candidates, key=lambda x: x["adjusted_score"], reverse=True)

    def evidence_angle(cand):
        full = ((cand.get("title") or "") + " " + (cand.get("text") or "")).lower()
        if any(t in full for t in ["eligibility criteria", "prescribing menopausal hormone therapy", "prescription of menopausal hormone therapy", "consensus document"]):
            return "hormone_prescribing"
        if any(t in full for t in ["fezolinetant", "elinzanetant", "neurokinin"]):
            return "nonhormonal_drug"
        if any(t in full for t in ["hormone therapy", "menopausal hormone therapy", "mht", "hrt", "estrogen therapy"]):
            return "hormone_therapy"
        if any(t in full for t in ["mindfulness", "cbt", "cognitive behavioral", "exercise", "acupuncture", "lifestyle"]):
            return "behavioral_lifestyle"
        if any(t in full for t in ["cardiovascular risk", "coronary heart disease", "cardiometabolic", "vascular health", "vascular aging"]):
            return "cardio_risk"
        if any(t in full for t in ["amenorrhea", "amenorrhoea", "menstrual", "final menstrual period", "clinical diagnosis"]):
            return "diagnosis"
        if any(t in full for t in ["prevalence", "quality of life", "menopause rating scale", "menqol"]):
            return "qol_prevalence"
        return "general"

    results = []
    seen_docs = {}
    seen_angles = {}

    for cand in candidates:
        doc_id = cand["source_id"]
        count = seen_docs.get(doc_id, 0)

        if count >= max_per_doc:
            continue

        angle = evidence_angle(cand)
        angle_count = seen_angles.get(angle, 0)

        if (treatment_query or cardiovascular_risk_query) and angle_count >= 2 and len(results) < max(3, k - 1):
            continue

        if cardiovascular_risk_query and angle == "hormone_prescribing" and any(
            evidence_angle(r) == "cardio_risk" for r in results
        ):
            continue

        cand["evidence_angle"] = angle
        results.append(cand)
        seen_docs[doc_id] = count + 1
        seen_angles[angle] = angle_count + 1

        if len(results) >= k:
            break

    return results


# 6) Test retrieval rápido
res = retrieve(
    "¿Qué puedo hacer con los sofocos durante la menopausia?",
    k=8,
    initial_k=20,
    max_per_doc=1,
    use_type_priority=True
)

for r in res:
    print(
        f"score={r['score']:.3f} | adj={r['adjusted_score']:.3f} | "
        f"type={r['type']} | pmid={r['pmid']} | source={r['source_id']}"
    )
    print("title:", r["title"])
    print(r["text"][:220], "\n---\n")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Embeddings shape: (1007, 384)
FAISS index size: 1007
score=0.515 | adj=0.585 | type=meta-analysis | pmid=42095813 | source=meta_42095813
title: [Clinical update and comprehensive approach to menopausal symptoms for healthcare professionals].
[Clinical update and comprehensive approach to menopausal symptoms for healthcare professionals]. Menopause is a physiological stage in a woman's life characterized by the permanent cessation of menstruation and decrease 
---

score=0.435 | adj=0.535 | type=included-study-candidate | pmid=14412840 | source=included_14412840
title: Contemporary therapy of the menopausal syndrome.
Contemporary therapy of the menopausal syndrome. 
---

score=0.406 | adj=0.526 | type=meta-analysis | pmid=42160056 | source=meta_42160056
title: Optimal Dietary Patterns for Lower Weight Gain and Risk of Obesity Surrounding Menopause.
ositive correlations with red or processed meats, sodium, and French fries. PHDI showed the largest positive correlations with nuts, unsatur

In [ ]:
# Pruebas pre-modelo

queries = [
    "How does CBT-I affect sleep quality in menopausal women?",
    "Does CBT-I improve insomnia severity during menopause?",
    "What evidence exists for CBT-I in postmenopausal women with vasomotor symptoms?"
]

for q in queries:
    print("\nQUERY:", q)
    res = retrieve(q, k=5, initial_k=15, max_per_doc=1, use_type_priority=True)
    for r in res:
        print(f"score={r['score']:.3f} | type={r['type']} | pmid={r['pmid']}")
        print("title:", r["title"])
        print("---")



QUERY: How does CBT-I affect sleep quality in menopausal women?
score=0.711 | type=meta-analysis | pmid=41531400
title: Effects of cognitive behavioral therapy on sleep quality and insomnia severity index in women with menopausal insomnia: a systematic review and meta-analysis.
---
score=0.758 | type=meta-analysis | pmid=39145901
title: Prevalence of poor sleep quality during menopause: a meta-analysis.
---
score=0.696 | type=meta-analysis | pmid=42090940
title: Effectiveness of non-pharmacological interventions for insomnia related to natural menopause: A meta-analysis of randomized controlled trials.
---
score=0.742 | type=meta-analysis | pmid=41713169
title: Cognitive-behavioral interventions for (peri)menopausal symptoms: A scoping review of interventions characteristics and content.
---
score=0.687 | type=meta-analysis | pmid=40575963
title: Sleep quality in perimenopausal and postmenopausal women: which exercise therapy is the most effective? A systematic review and network meta

10) Generative model: Llama Libre


10.1) Dependencias para Llama Transformers


In [ ]:
# Ejecutar una vez antes de cargar Llama.
# En Colab/Jupyter, %pip instala en el kernel activo.
%pip install -q transformers accelerate bitsandbytes sentencepiece huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00


10.2) Login Hugging Face para modelos gated


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
model_name = MODEL_NAME

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

def load_llama_model(model_id):
    tokenizer_obj = AutoTokenizer.from_pretrained(model_id)

    model_obj = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    )

    if tokenizer_obj.pad_token is None:
        tokenizer_obj.pad_token = tokenizer_obj.eos_token

    model_obj.eval()
    return tokenizer_obj, model_obj

try:
    tokenizer, model = load_llama_model(MODEL_NAME)
    model_name = MODEL_NAME
    print("Loaded:", MODEL_NAME)
except Exception as model_error:
    raise RuntimeError(
        "No se pudo cargar Llama. Si aparece 403/gated repo, acepta la licencia "
        "en Hugging Face y ejecuta la celda de login antes de cargar el modelo. "
        f"Error original: {model_error}"
    ) from model_error


Device: cuda


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Loaded: meta-llama/Llama-3.1-8B-Instruct


11) Full RAG (build_prompt + answer_rag)

In [ ]:
import torch, re

def build_context(results, max_chars=3200, max_per_doc=1, question=None, question_intent="general"):
    """
    Construye el contexto para RAG reduciendo ruido del abstract.
    Mantiene título, PMID, tipo y selecciona frases según la intención.
    """
    context_parts = []
    total_chars = 0
    seen_docs = {}

    q = (question or "").lower()
    question_intent = question_intent or "general"

    useful_markers = [
        "aim", "objective", "purpose",
        "result", "results",
        "conclusion", "conclusions",
        "found", "showed", "demonstrated",
        "improved", "reduced", "increased",
        "associated", "effective", "efficacy",
        "symptoms", "treatment", "therapy",
        "quality of life", "sleep", "insomnia",
        "hot flashes", "hot flushes", "vasomotor",
        "depression", "anxiety",
        "diagnosis", "clinical", "management",
        "menopause", "menopausal", "perimenopause"
    ]

    intent_markers = {
        "diagnosis": [
            "diagnosis", "clinical", "menopausal transition", "perimenopause",
            "menstrual", "amenorrhea", "amenorrhoea", "irregular periods",
            "12 months", "final menstrual period", "hot flashes",
            "night sweats", "vaginal dryness", "genitourinary"
        ],
        "cause": [
            "cause", "causes", "mechanism", "mechanisms", "pathophysiology",
            "thermoregulatory", "thermoregulation", "hypothalamic",
            "neurokinin", "estrogen", "vasomotor", "hot flashes",
            "hot flushes", "night sweats"
        ],
        "treatment": [
            "treatment", "therapy", "management", "intervention",
            "effective", "efficacy", "improved", "reduced", "hormone therapy",
            "nonhormonal", "cbt", "mindfulness", "exercise", "fezolinetant",
            "elinzanetant"
        ],
        "prognosis": [
            "duration", "persist", "persistence", "course", "trajectory",
            "longitudinal", "transition", "years", "temporary"
        ],
        "risk": [
            "cardiovascular risk", "cardiometabolic risk", "coronary heart disease",
            "vascular health", "vascular aging", "atherosclerosis",
            "blood pressure", "lipid", "metabolic", "estrogen decline",
            "traditional risk factors", "nontraditional risk factors"
        ],
        "practical": [
            "guideline", "recommendation", "clinical", "assessment",
            "screening", "follow-up", "management", "healthcare"
        ],
        "monitoring": [
            "guideline", "recommendation", "clinical", "assessment",
            "screening", "follow-up", "diagnosis", "blood test",
            "laboratory", "bone mineral density", "dxa"
        ],
        "prevention": [
            "prevention", "preventive", "lifestyle", "exercise",
            "physical activity", "diet", "cardiovascular risk",
            "bone health", "weight", "smoking", "alcohol"
        ],
        "emotional": [
            "quality of life", "experience", "perceptions",
            "wellbeing", "mental health", "anxiety", "depression",
            "psychological"
        ]
    }

    noise_markers = [
        "database", "databases",
        "searched", "search strategy",
        "heterogeneity",
        "publication bias",
        "prisma",
        "protocol",
        "registration",
        "confidence interval",
        "ci ",
        "i2",
        "p-value",
        "subgroup analysis"
    ]

    def split_sentences(text):
        text = text.replace("\n", " ")
        text = re.sub(r"\s+", " ", text).strip()
        return re.split(r'(?<=[.!?])\s+', text)

    def sentence_score(sentence):
        low = sentence.lower()
        score = 0

        score += sum(2 for marker in intent_markers.get(question_intent, []) if marker in low)
        score += sum(1 for marker in useful_markers if marker in low)

        topic_terms = [
            "menopause", "menopausal", "perimenopause", "postmenopausal",
            "hot flashes", "hot flushes", "vasomotor", "sleep", "insomnia",
            "depression", "anxiety", "cardiovascular", "hormone"
        ]
        score += sum(1 for marker in topic_terms if marker in q and marker in low)

        return score

    def compress_text(text, max_sentences=5):
        if not text:
            return ""

        sentences = split_sentences(text)
        candidates = []

        for pos, s in enumerate(sentences):
            s_clean = s.strip()
            low = s_clean.lower()

            if len(s_clean) < 40:
                continue

            if any(n in low for n in noise_markers):
                continue

            score = sentence_score(s_clean)
            if score > 0:
                candidates.append((score, pos, s_clean))

        if candidates:
            candidates = sorted(candidates, key=lambda x: (-x[0], x[1]))
            selected = sorted(candidates[:max_sentences], key=lambda x: x[1])
            return " ".join(s for _, _, s in selected)

        selected = []
        for s in sentences:
            s_clean = s.strip()
            if len(s_clean) >= 40:
                selected.append(s_clean)
            if len(selected) >= 3:
                break

        return " ".join(selected)

    for r in results:
        doc_id = r.get("source_id")

        count = seen_docs.get(doc_id, 0)
        if count >= max_per_doc:
            continue

        title = r.get("title", "Unknown title")
        pmid = r.get("pmid", "NA")
        source_type = r.get("type", "unknown")
        text = r.get("text", "")

        compressed = compress_text(text, max_sentences=5)

        if not compressed:
            continue

        piece = (
            f"[TITLE] {title}\n"
            f"[PMID] {pmid}\n"
            f"[TYPE] {source_type}\n"
            f"[TEXT] {compressed}\n"
        )

        if total_chars + len(piece) > max_chars:
            break

        context_parts.append(piece)
        total_chars += len(piece)
        seen_docs[doc_id] = count + 1

    return "\n\n".join(context_parts)


def expand_query_for_retrieval(question):
    q = question.lower()
    extra_terms = []
    intent = classify_intent(question) if "classify_intent" in globals() else "general"

    if intent != "digital_orientation" and any(term in q for term in [
        "cómo sé", "como se",
        "saber si", "estoy en menopausia",
        "es menopausia", "menopausia o no",
        "perimenopausia o no",
        "diagnóstico", "diagnostico",
        "diagnosticar", "confirmar menopausia"
    ]):
        extra_terms.append(
            "menopause diagnosis clinical diagnosis menopausal transition "
            "perimenopause 12 months amenorrhea amenorrhoea final menstrual period "
            "menstrual irregularity irregular periods vasomotor symptoms "
            "hot flashes hot flushes night sweats vaginal dryness genitourinary symptoms"
        )

    if any(term in q for term in [
        "cbt", "cbt-i", "cognitive behavioral", "cognitive behavioural",
        "insomnia", "sleep", "sueño", "dormir", "duermo", "insomnio",
        "desvelo", "me despierto", "calidad del sueño",
        "terapia cognitivo-conductual", "terapia cognitivo conductual",
        "tcc", "tcc-i", "conductual", "cognitivo-conductual"
    ]):
        extra_terms.append(
            "CBT-I CBT cognitive behavioral therapy cognitive behavioural therapy "
            "CBT-Meno insomnia sleep quality sleep disturbance insomnia severity "
            "menopausal insomnia menopausal symptoms menopause symptoms "
            "menopausal women postmenopausal women perimenopausal women "
            "ISI PSQI sleep hygiene sleep restriction therapy"
        )

    if any(term in q for term in [
        "hot flash", "hot flashes", "hot flush", "hot flushes",
        "vasomotor", "sofoco", "sofocos", "calor", "sudor", "sudores",
        "sudoración", "night sweats", "sudores nocturnos"
    ]):
        if intent == "cause":
            extra_terms.append(
                "hot flashes hot flushes vasomotor symptoms night sweats "
                "thermoregulatory dysfunction thermoregulation hypothalamic "
                "neurokinin estrogen decline pathophysiology mechanism menopause transition"
            )
        elif intent == "treatment":
            extra_terms.append(
                "hot flashes hot flushes vasomotor symptoms night sweats "
                "menopause treatment management hormone therapy nonhormonal therapy "
                "fezolinetant elinzanetant CBT mindfulness exercise acupuncture"
            )
        else:
            extra_terms.append(
                "hot flashes hot flushes vasomotor symptoms night sweats "
                "menopause perimenopause postmenopause menopausal symptoms "
                "mechanism treatment management"
            )

    if any(term in q for term in [
        "mindfulness", "mbsr", "meditación", "meditacion", "meditation",
        "atención plena", "atencion plena"
    ]):
        extra_terms.append(
            "mindfulness MBSR mindfulness-based stress reduction meditation "
            "mind-body therapies menopausal symptoms quality of life "
            "anxiety depression sleep quality postmenopausal women"
        )

    if any(term in q for term in [
        "mental health", "depression", "depressive", "anxiety", "stress",
        "psychological", "mood", "irritable", "irritability",
        "salud mental", "depresión", "depresion", "ansiedad", "estrés",
        "estres", "ánimo", "animo", "irritable", "triste", "emocional"
    ]):
        extra_terms.append(
            "menopause mental health depression anxiety stress psychological symptoms "
            "mood irritability depressive symptoms postmenopausal women "
            "perimenopausal women quality of life mindfulness CBT mind-body therapies"
        )

    if any(term in q for term in [
        "quality of life", "wellbeing", "well-being", "symptoms",
        "calidad de vida", "bienestar", "síntomas", "sintomas",
        "normal", "etapa", "menopausia", "perimenopausia", "postmenopausia"
    ]):
        if intent in ["general", "emotional"]:
            extra_terms.append(
                "menopause menopausal symptoms climacteric symptoms quality of life "
                "Menopause Rating Scale MRS MENQOL vasomotor psychological physical sexual symptoms "
                "perimenopausal women postmenopausal women"
            )

    if any(term in q for term in [
        "cognition", "cognitive", "memory", "brain fog", "concentration",
        "memoria", "concentración", "concentracion", "niebla mental",
        "despistes", "olvidos", "pensar", "lenta", "cabeza"
    ]):
        extra_terms.append(
            "menopause cognition cognitive function memory brain fog concentration "
            "executive function cognitive complaints perimenopause postmenopause"
        )

    if any(term in q for term in [
        "skin", "dryness", "weight", "body", "fat", "hair",
        "piel", "sequedad", "peso", "engorda", "barriga",
        "cuerpo", "físico", "fisico", "pelo", "calva"
    ]):
        extra_terms.append(
            "menopause physical symptoms body composition weight gain abdominal fat "
            "skin dryness hair loss postmenopausal women estrogen decline"
        )

    if any(term in q for term in [
        "hormone therapy", "hormonal therapy", "mht", "hrt",
        "medication", "supplement", "cancer risk",
        "terapia hormonal", "tratamiento hormonal", "medicación", "medicacion",
        "suplementos", "cáncer", "cancer", "medicamentos"
    ]):
        extra_terms.append(
            "menopause hormone therapy menopausal hormone therapy MHT HRT "
            "nonhormonal treatment medication supplements cancer risk "
            "clinical guidelines postmenopausal women"
        )

    if any(term in q for term in [
        "cardiovascular", "cardio", "corazón", "corazon", "heart",
        "enfermedad cardiovascular", "riesgo cardiovascular"
    ]):
        extra_terms.append(
            "menopause cardiovascular disease heart disease cardiometabolic risk "
            "vascular health blood pressure lipid profile postmenopausal women estrogen decline "
            "menopause transition cardiovascular risk vascular aging"
        )

    if any(term in q for term in [
        "prevenir", "prevención", "prevencion", "problemas futuros",
        "estilo de vida", "hábitos", "habitos", "debería cambiar",
        "deberia cambiar", "actividad física", "actividad fisica",
        "alimentación", "alimentacion", "dieta"
    ]):
        extra_terms.append(
            "menopause prevention lifestyle physical activity exercise diet "
            "cardiovascular risk bone health weight management healthy aging "
            "clinical recommendations postmenopausal women"
        )

    ia_patterns = [
        r"\bapp\b",
        r"\bapps\b",
        r"\bai\b",
        r"\bia\b",
        r"\binteligencia artificial\b",
        r"\bartificial intelligence\b",
        r"\btechnology\b",
        r"\btecnología\b",
        r"\btecnologia\b",
        r"\bdigital\b"
    ]

    if any(re.search(pattern, q) for pattern in ia_patterns):
        extra_terms.append(
            "digital health artificial intelligence health app menopause symptom tracking "
            "clinical decision support patient education medical advice diagnosis limitations"
        )

    if not extra_terms:
        if intent in ["monitoring", "digital_orientation"]:
            extra_terms.append(
                "symptom tracking menopause symptom diary patient education shared decision making "
                "clinical consultation quality of life menopause assessment health guidance "
                "self-monitoring patient reported outcomes Menopause Rating Scale MENQOL "
                "digital health mobile app clinical decision support"
            )
        elif intent == "emotional":
            extra_terms.append(
                "menopause experience menopause perceptions quality of life emotional symptoms "
                "psychological symptoms mental health anxiety depression patient experience "
                "menopause transition supportive care"
            )
        else:
            extra_terms.append(
                "menopause menopausal symptoms climacteric symptoms perimenopause postmenopause "
                "vasomotor symptoms hot flashes night sweats sleep mood quality of life "
                "symptom tracking patient education clinical consultation health guidance"
            )

    if extra_terms:
        return question + " " + " ".join(extra_terms)

    return question

def generate_answer(prompt, max_new_tokens=220):
    max_new_tokens = max(max_new_tokens, 220)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
        return_dict=True
    )

    if isinstance(inputs, torch.Tensor):
        input_ids = inputs.to(model.device)
        attention_mask = torch.ones_like(input_ids)
    else:
        input_ids = inputs["input_ids"].to(model.device)
        attention_mask = inputs.get("attention_mask")
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        else:
            attention_mask = attention_mask.to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][input_ids.shape[-1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()


import re

def clean_answer(text):
    if not text:
        return "No hay suficiente información relevante en el contexto recuperado."

    bad_tokens = [
        "<|assistant|>", "<|user|>", "<|system|>",
        "[/INST]", "[INST]", "<s>", "</s>"
    ]

    for tok in bad_tokens:
        text = text.replace(tok, " ")

    text = text.replace("**", " ")
    text = text.replace("*", " ")
    text = text.replace("[TITLE]", "el estudio")
    text = text.replace("[Título]", "el estudio")
    text = text.replace("[PMID]", "PMID")
    text = text.replace("[TYPE]", "tipo")
    text = text.replace("[TEXT]", "texto")
    text = text.strip()

    section_markers = [
        "Answer in English", "Answer in Spanish",
        "Responde en español", "Respuesta:", "Answer:"
    ]

    for marker in section_markers:
        if marker in text:
            text = text.split(marker)[-1].strip()

    leading_markers = [
        "Pregunta del usuario:", "Pregunta:", "User question:",
        "Question:", "Contexto científico:", "Contexto:",
        "Scientific context:", "Context:"
    ]

    for marker in leading_markers:
        text = text.replace(marker, " ")

    meta_patterns = [
        r"^nota:\s*",
        r"^explicaci[oó]n:\s*",
        r"^en base al contexto proporcionado,?\s*",
        r"^en base al contexto,?\s*",
        r"^bas[aá]ndose en el contexto proporcionado,?\s*",
        r"^seg[uú]n el contexto proporcionado,?\s*",
        r"^seg[uú]n el contexto,?\s*",
        r"^de acuerdo con el contexto proporcionado,?\s*",
        r"^con base en el contexto proporcionado,?\s*",
        r"^el texto proporciona informaci[oó]n sobre\s*",
        r"^el contexto proporciona informaci[oó]n sobre\s*",
        r"^el contexto indica que\s*",
        r"^la respuesta se basa en el contexto proporcionado\.?\s*",
        r"^la respuesta se basa en el contexto\.?\s*",
        r"^la respuesta es que\s*",
        r"^la respuesta es\s*",
        r"^en respuesta a la pregunta,?\s*",
        r"^para responder a la pregunta,?\s*",
        r"^based on the provided context,?\s*",
        r"^according to the provided context,?\s*",
    ]

    for pat in meta_patterns:
        text = re.sub(pat, "", text, flags=re.IGNORECASE).strip()

    text = re.sub(r"\bEn base al contexto proporcionado,?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\bEn base al contexto,?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\bEl texto proporciona informaci[oó]n sobre\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\bLa informaci[oó]n proporcionada no ofrece informaci[oó]n sobre\b", "El contexto recuperado no ofrece información específica sobre", text, flags=re.IGNORECASE)

    stop_markers = [
        "```json", "```", "Prioridades:", "Limitaciones:", "Recomendaciones:",
        "Importante:", "Let me know", "Would you like", "Please note",
        "Nota:", "Nota: Esta respuesta", "Nota: La respuesta",
        "Esta información no sustituye"
    ]
    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0].strip()

    lines = [l.strip() for l in text.split("\n") if l.strip()]
    clean_lines = []

    for line in lines:
        low = line.lower().strip()

        if re.fullmatch(r"[\*\-\#\:\s]+", line):
            continue

        if low in [
            "contexto", "respuesta", "answer", "context", "question",
            "pregunta", "explicación", "explicacion", "nota"
        ]:
            continue

        if line.endswith("?") and len(line) < 220:
            continue

        clean_lines.append(line)

    text = " ".join(clean_lines).strip()

    replacements = {
        "insomia": "insomnio",
        "insomno": "insomnio",
        "insomni": "insomnio",
        "insomnio premenstrual": "insomnio relacionado con la menopausia",
        "premenopausico": "menopáusico",
        "menopausica": "menopáusica",
        "menopausico": "menopáusico",
        "menopausicas": "menopáusicas",
        "menopausicos": "menopáusicos",
        "perimenopausicas": "perimenopáusicas",
        "soffoces": "sofocos",
        "sofocones": "sofocos",
        "sudores nocturnas": "sudores nocturnos",
        "sudores nocturnes": "sudores nocturnos",
        "menoapausia": "menopausia",
        "menópausica": "menopáusica",
        "menópausico": "menopáusico",
        "menopausa": "menopausia",
        "menopauasa": "menopausia",
        "menopauza": "menopausia",
        "menopausiá": "menopausia",
        "menopausía": "menopausia",
        "menopausing": "menopausia",
        "postmenopanúsicas": "postmenopáusicas",
        "postmenopausicas": "postmenopáusicas",
        "qualité": "calidad",
        "calidad del dormir": "calidad del sueño",
        "calidad de sueño": "calidad del sueño",
        "toda la mujer": "todas las mujeres",
        "todos las mujeres": "todas las mujeres",
        "menopause": "menopausia",
        "empeorara": "empeorar",
        "médicamentariamente": "médicamente",
        "médicament": "médicamente",
        "cognitivos": "cognitivo"
    }

    for bad, good in replacements.items():
        text = re.sub(rf"\b{bad}\b", good, text, flags=re.IGNORECASE)

    text = re.sub(r"\bla insomnio\b", "el insomnio", text, flags=re.IGNORECASE)
    text = re.sub(r"\bde el insomnio\b", "del insomnio", text, flags=re.IGNORECASE)
    text = re.sub(r"\bgravedad de el insomnio\b", "gravedad del insomnio", text, flags=re.IGNORECASE)
    text = re.sub(r"\bseveridad de el insomnio\b", "gravedad del insomnio", text, flags=re.IGNORECASE)
    text = re.sub(r"\binsomnio relacionada\b", "insomnio relacionado", text, flags=re.IGNORECASE)
    text = re.sub(r"\bmujeres menopáusica\b", "mujeres menopáusicas", text, flags=re.IGNORECASE)
    text = re.sub(r"\bmujeres menopáusico\b", "mujeres menopáusicas", text, flags=re.IGNORECASE)
    text = re.sub(r"\binsomnio\s+premenstrual\b", "insomnio relacionado con la menopausia", text, flags=re.IGNORECASE)
    text = re.sub(r"\bmujeres con insomnio premenstrual\b", "mujeres con insomnio relacionado con la menopausia", text, flags=re.IGNORECASE)
    text = re.sub(r"\btratamiento m[aá]s efectivo\b", "una de las opciones con mejor evidencia recuperada", text, flags=re.IGNORECASE)
    text = re.sub(r"\bel mejor tratamiento\b", "una de las opciones con mejor evidencia recuperada", text, flags=re.IGNORECASE)
    text = re.sub(r"\bEl una de las opciones\b", "Una de las opciones", text, flags=re.IGNORECASE)
    text = re.sub(r"\bla una de las opciones\b", "una de las opciones", text, flags=re.IGNORECASE)
    text = re.sub(r"\bes eficaz para tratar\b", "puede ser eficaz para tratar", text, flags=re.IGNORECASE)
    text = re.sub(r"\bha demostrado ser efectiva\b", "ha mostrado evidencia de eficacia", text, flags=re.IGNORECASE)
    text = re.sub(r"\bhan demostrado\b", "sugieren", text, flags=re.IGNORECASE)
    text = re.sub(r"\bLa menopausia puede afectar negativamente la salud mental\b", "La menopausia puede asociarse con cambios en la salud mental", text, flags=re.IGNORECASE)
    text = re.sub(r"\bcambios hormonales que pueden afectar negativamente la salud mental\b", "cambios hormonales que pueden asociarse con síntomas de salud mental", text, flags=re.IGNORECASE)
    text = re.sub(r"\bpuede aumentar el riesgo de estos trastornos\b", "se ha asociado con mayor vulnerabilidad a estos síntomas", text, flags=re.IGNORECASE)
    text = re.sub(r"\bLa disminución de estrógeno durante la menopausia puede aumentar el riesgo de depresión, ansiedad y deterioro cognitivo\b", "Los cambios hormonales de la menopausia se han asociado con mayor vulnerabilidad a síntomas depresivos, ansiosos o cognitivos", text, flags=re.IGNORECASE)
    text = re.sub(r"\bLa disminución de estrógeno durante la menopausia puede aumentar el riesgo de estos trastornos\b", "Los cambios hormonales de la menopausia se han asociado con mayor vulnerabilidad a estos síntomas", text, flags=re.IGNORECASE)
    text = re.sub(r"\bEstudios sugieren que la disminución de estrógeno durante la menopausia puede aumentar el riesgo de depresión, ansiedad y deterioro cognitivo\.?\s*", "Estudios sugieren que los cambios hormonales de la menopausia se han asociado con mayor vulnerabilidad a síntomas depresivos, ansiosos o cognitivos. ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bpuede aumentar la vulnerabilidad a la depresión y otros trastornos mentales\b", "puede asociarse con mayor vulnerabilidad a síntomas depresivos o ansiosos", text, flags=re.IGNORECASE)
    text = re.sub(r"\bEl estrés puede empeorar los síntomas de menopausia\b", "El estrés puede contribuir a empeorar algunos síntomas durante la menopausia", text, flags=re.IGNORECASE)
    text = re.sub(r"\bSí, El estrés\b", "Sí, el estrés", text)
    text = re.sub(r"\bse puede inferir que\b", "los documentos sugieren de forma indirecta que", text, flags=re.IGNORECASE)
    text = re.sub(r"\bLa intervención temprana puede ser beneficiosa\b", "Puede ser razonable abordar los factores modificables y consultar si los síntomas empeoran", text, flags=re.IGNORECASE)
    text = re.sub(r"\bBasado en la evidencia disponible,\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\bque Puede\b", "que puede", text)
    text = re.sub(r"\bque Los cambios hormonales\b", "que los cambios hormonales", text)
    text = re.sub(r"\bmejorar la calidad del descanso\b", "mejorar la calidad del sueño", text, flags=re.IGNORECASE)
    text = re.sub(r"\bseveridad del insomnio\b", "gravedad del insomnio", text, flags=re.IGNORECASE)
    text = re.sub(r"\bdeterioro cognitivo\.\s*El uso de terapia hormonal", "deterioro cognitivo. El uso de terapia hormonal", text, flags=re.IGNORECASE)
    if re.search(r"\b(CBT-I|CBT-i|CBT|TCC|terapia cognitivo[- ]conductual)\b", text, flags=re.IGNORECASE):
        text = re.sub(r"\s*Además,\s+el ejercicio físico regular ha demostrado ser beneficioso[^.]*\.", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*Además,\s+diversos estudios han encontrado que el ejercicio físico regular puede mejorar[^.]*\.", "", text, flags=re.IGNORECASE)
    text = re.sub(r"[^\x00-\x7FáéíóúÁÉÍÓÚñÑüÜ¿¡.,;:()/%\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    text = re.sub(r"\.{2,}", ".", text)
    text = text.strip(" -:;,.")
    text = text.strip()

    sentences = re.split(r'(?<=[.!?])\s+', text)
    seen = set()
    unique = []

    for s in sentences:
        s = s.strip()
        if not s:
            continue
        key = re.sub(r"\s+", " ", s.lower())
        if key not in seen:
            seen.add(key)
            unique.append(s)

    text = " ".join(unique).strip()

    cut_endings = [
        " post", " post.", " mujeres post", " mujeres post.",
        " mujeres en", " mujeres con", " durante", " en mujeres post"
    ]

    lower_text = text.lower()
    if any(lower_text.endswith(end) for end in cut_endings):
        parts = re.split(r'(?<=[.!?])\s+', text)
        if len(parts) > 1:
            text = " ".join(parts[:-1]).strip()

    parts = re.split(r'(?<=[.!?])\s+', text)
    if len(parts) > 1 and len(parts[-1].split()) <= 3:
        text = " ".join(parts[:-1]).strip()

    if len(text) < 25:
        return "No hay suficiente información relevante en el contexto recuperado."

    if re.fullmatch(r"[\W_]+", text):
        return "No hay suficiente información relevante en el contexto recuperado."

    if text and text[-1] not in ".!?":
        text += "."

    copied_context_patterns = [
        r"\bPMID\s*\d+",
        r"\btipo\s+meta-analysis\b",
        r"\btexto\s+Non-Hormonal\b",
        r"\bel estudio\s+Non-Hormonal Drug Treatment\b",
        r"\bMSs\.?\s+el estudio\b",
        r"\b\[?\d{7,8}\]?\b"
    ]

    copied_hits = sum(1 for pat in copied_context_patterns if re.search(pat, text, flags=re.IGNORECASE))
    if copied_hits >= 2:
        return "No hay suficiente información relevante en el contexto recuperado."

    if re.match(r"^(científico|cientifica|científica),\s*", text, flags=re.IGNORECASE):
        text = re.sub(r"^(científico|cientifica|científica),\s*", "", text, flags=re.IGNORECASE).strip()

    truncated_endings = [
        "que regulan.", "que regulan", "se relaciona con.", "se debe a.",
        "puede deberse a.", "está asociado con.", "incluyendo."
    ]
    if any(text.lower().endswith(end) for end in truncated_endings):
        return "No hay suficiente información relevante en el contexto recuperado."

    text = text[0].upper() + text[1:]

    return text

# Clasificador básico
def classify_question(question):
    q = question.lower()

    digital_terms = [
        "app", "apps", "aplicación", "aplicacion",
        "inteligencia artificial", "tecnología", "tecnologia",
        "orientarme", "sin diagnosticar", "seguimiento diario",
        "registrar síntomas", "registrar sintomas", "usar tecnología",
        "usar tecnologia", "sin sustituir al médico", "sin sustituir al medico",
        "qué puede hacer una ia", "que puede hacer una ia",
        "qué no puede hacer una ia", "que no puede hacer una ia"
    ]
    q_words = re.findall(r"\b\w+\b", q)

    if any(t in q for t in digital_terms) or "ia" in q_words:
        return "practical_guidance"

    emotional_support_keywords = [
        "estoy exagerando", "realmente me está pasando algo",
        "realmente me esta pasando algo", "nadie me explica",
        "voy a ciegas", "me siento perdida", "me siento perdido",
        "me preocupa", "me agobia", "me da miedo", "me asusta",
        "no soy la misma", "no soy el mismo", "acompañamiento",
        "acompanamiento", "me siento sola", "me siento solo",
        "no me entienden", "me frustra", "frustrada", "frustrado"
    ]

    practical_guidance_keywords = [
        "qué pruebas", "que pruebas", "qué analíticas", "que analiticas",
        "análisis de sangre", "analisis de sangre", "qué especialista",
        "que especialista", "a qué especialista", "a que especialista",
        "qué médico", "que medico", "qué médica", "que medica",
        "a quién consulto", "a quien consulto", "a quién debería consultar",
        "a quien deberia consultar", "qué seguimiento", "que seguimiento",
        "qué controles", "que controles", "revisiones",
        "densitometría", "densitometria", "bien controlada",
        "controlada médicamente", "controlada medicamente",
        "qué debería revisar", "que deberia revisar",
        "profesional de la salud", "médico", "medico",
        "ginecólogo", "ginecologo", "endocrino", "endocrinólogo",
        "endocrinologo", "consulta", "derivar", "derivación", "derivacion",
        "organizar toda la información", "organizar toda la informacion",
        "organizar la información", "organizar la informacion",
        "qué información necesito", "que informacion necesito",
        "tomar decisiones sobre mi salud", "historial", "registro de síntomas",
        "registro de sintomas", "llevar a la consulta",
        "visión global", "vision global", "mi estado",
        "qué es importante", "que es importante",
        "qué no", "que no", "guía clara", "guia clara",
        "seguimiento diario", "forma de tener seguimiento",
        "tener una guía", "tener una guia",
        "qué hacer en cada momento", "que hacer en cada momento",
        "vitamina", "vitaminas", "suplemento", "suplementos",
        "análisis de sangre", "analisis de sangre",
        "qué debería haber sabido", "que deberia haber sabido",
        "a qué edad", "a que edad",
        "evitar empeorar", "para evitar empeorar",
        "prevenir", "prevención", "prevencion", "problemas futuros",
        "qué debería haber hecho", "que deberia haber hecho",
        "estilo de vida", "hábitos", "habitos", "debería cambiar",
        "deberia cambiar", "actividad física", "actividad fisica",
        "alimentación", "alimentacion", "dieta"
    ]

    clinical_uncertainty_keywords = [
        "cómo sé", "como se", "saber si", "esto que me pasa",
        "lo que me pasa", "me está pasando", "me esta pasando",
        "menopausia o no", "es menopausia", "estoy en menopausia",
        "perimenopausia o no", "puede ser solo estrés",
        "puede ser solo estres", "todo lo que me pasa",
        "hormonal o no", "normal en esta etapa", "debería preocuparme",
        "deberia preocuparme", "sin tener todos los síntomas",
        "sin tener todos los sintomas", "certeza", "relacionando mal",
        "puede ser menopausia", "será menopausia", "sera menopausia"
    ]

    limitation_keywords = [
        "app", "apps", "inteligencia artificial",
        "piel", "picores", "picor", "inmunológico", "inmunologico",
        "calva", "pelo", "cabello", "famosas",
        "madre", "abuela", "familia de mi padre",
        "genética", "genetica"
    ]

    scientific_keywords = [
        "cbt", "cbt-i", "tcc", "tcc-i",
        "terapia cognitivo-conductual", "terapia cognitivo conductual",
        "mindfulness", "mbsr", "qigong", "yoga", "pilates",
        "ejercicio", "exercise", "terapia", "tratamiento", "treatment",
        "isoflavonas", "soja", "soy", "sofocos", "calor",
        "hot flashes", "hot flushes", "vasomotor", "sudores nocturnos",
        "insomnio", "sueño", "sleep", "insomnia", "calidad del sueño",
        "salud mental", "mental health", "depresión", "depresion",
        "depression", "ansiedad", "anxiety", "estrés", "estres",
        "stress", "irritabilidad", "irritable", "estado emocional",
        "emocional", "ánimo", "animo", "calidad de vida",
        "quality of life", "menopausia", "perimenopausia",
        "postmenopausia", "menopause", "perimenopause",
        "postmenopause", "cardiovascular", "corazón", "corazon",
        "libido", "sequedad vaginal", "terapia hormonal",
        "tratamiento hormonal"
    ]

    if any(kw in q for kw in emotional_support_keywords):
        return "emotional_support"

    if any(kw in q for kw in [
        "vitamina", "vitaminas", "suplemento", "suplementos",
        "análisis de sangre", "analisis de sangre",
        "analítica", "analitica", "analíticas", "analiticas",
        "bien controlada", "controlada médicamente", "controlada medicamente",
        "cómo puedo saber si estoy bien", "como puedo saber si estoy bien",
        "médicamente", "medicamente"
    ]):
        return "practical_guidance"

    if any(kw in q for kw in clinical_uncertainty_keywords):
        return "clinical_uncertainty"

    if any(kw in q for kw in practical_guidance_keywords):
        return "practical_guidance"

    if any(kw in q for kw in limitation_keywords):
        return "limitation"

    if any(kw in q for kw in scientific_keywords):
        return "scientific"

    return "scientific"


def classify_intent(question):
    q = question.lower()

    digital_terms = [
        "app", "apps", "aplicación", "aplicacion",
        "inteligencia artificial", "tecnología", "tecnologia",
        "orientarme", "sin diagnosticar", "seguimiento diario",
        "registrar síntomas", "registrar sintomas", "usar tecnología",
        "usar tecnologia", "sin sustituir al médico", "sin sustituir al medico",
        "qué puede hacer una ia", "que puede hacer una ia",
        "qué no puede hacer una ia", "que no puede hacer una ia"
    ]

    q_words = re.findall(r"\b\w+\b", q)

    if any(t in q for t in digital_terms) or "ia" in q_words:
        return "digital_orientation"

    if any(t in q for t in [
        "riesgo cardiovascular", "enfermedad cardiovascular",
        "cardiovascular", "corazón", "corazon", "heart disease",
        "cardiometabolic", "cardiometabólico", "cardiometabolico",
        "aumenta el riesgo", "riesgo de enfermedad"
    ]) and not any(t in q for t in [
        "terapia hormonal", "tratamiento hormonal", "hormone therapy",
        "mht", "hrt"
    ]):
        return "risk"

    if any(t in q for t in [
        "prevenir", "prevención", "prevencion", "evitar",
        "problemas futuros", "futuro", "riesgo futuro", "estilo de vida",
        "hábitos", "habitos", "cambiar en mi vida", "debería cambiar",
        "deberia cambiar", "actividad física", "actividad fisica",
        "dieta", "alimentación", "alimentacion",
        "qué debería haber hecho", "que deberia haber hecho",
        "haber hecho antes", "qué debería haber sabido", "que deberia haber sabido",
        "a qué edad", "a que edad", "evitar empeorar", "para evitar empeorar"
    ]):
        return "prevention"

    if any(t in q for t in [
        "qué pruebas", "que pruebas", "analíticas", "analiticas",
        "especialista", "revisiones", "controles", "seguimiento",
        "consulta", "organizar", "qué información", "que informacion",
        "tomar decisiones", "registro de síntomas", "registro de sintomas",
        "densitometría", "densitometria", "screening", "cribado",
        "monitorizar", "monitorización", "monitorizacion",
        "visión global", "vision global", "mi estado",
        "qué es importante", "que es importante",
        "guía clara", "guia clara", "seguimiento diario",
        "qué hacer en cada momento", "que hacer en cada momento",
        "vitamina", "vitaminas", "suplemento", "suplementos"
    ]):
        return "monitoring"

    if any(t in q for t in [
        "bien controlada", "controlada médicamente", "controlada medicamente",
        "médicamente", "medicamente"
    ]):
        return "monitoring"

    if any(t in q for t in [
        "cómo sé", "como se", "saber si", "diagnóstico", "diagnostico",
        "diagnosticar", "confirmar", "es menopausia", "menopausia o no",
        "estoy en menopausia", "puede ser menopausia", "hormonal o no",
        "sin tener todos los síntomas", "sin tener todos los sintomas",
        "saber con certeza"
    ]):
        return "diagnosis"

    if any(t in q for t in [
        "qué puedo hacer", "que puedo hacer", "tratamiento", "tratar",
        "mejorar", "aliviar", "funciona", "opciones", "treatment",
        "therapy", "manage", "management", "manejo"
    ]):
        return "treatment"

    if any(t in q for t in [
        "por qué", "por que", "porque tengo", "a qué se debe", "a que se debe",
        "qué causa", "que causa", "why", "cause", "causes", "mecanismo"
    ]):
        return "cause"

    if any(t in q for t in [
        "cuánto dura", "cuanto dura", "para siempre", "se quedan",
        "temporal", "pasajero", "reversible", "evolución", "evolucion"
    ]):
        return "prognosis"

    if any(t in q for t in [
        "me siento", "me preocupa", "me agobia", "miedo", "asusta",
        "perdida", "perdido", "exagerando"
    ]):
        return "emotional"

    return "general"


def filter_contexts(question, contexts, question_type="scientific", min_score=0.55, question_intent=None):
    """
    Filtra contextos débiles o tangenciales antes de pasarlos al modelo.
    Usa también la intención para separar diagnóstico, causa y tratamiento.
    """
    if not contexts:
        return []

    q = question.lower()
    question_intent = question_intent or classify_intent(question)

    def has_any(text, terms):
        return any(t in text for t in terms)

    filtered = []

    for c in contexts:
        score = c.get("adjusted_score", c.get("score", 0.0))
        title = (c.get("title") or "").lower()
        text = (c.get("text") or "").lower()
        full = title + " " + text

        if score < min_score:
            continue

        if question_type == "clinical_uncertainty":

            mental_query = has_any(q, [
                "ánimo", "animo", "ansiedad", "depresión", "depresion",
                "estrés", "estres", "mental", "psicológico", "psicologico",
                "me siento", "triste", "irritable"
            ])

            diagnosis_query = question_intent == "diagnosis" or has_any(q, [
                "cómo sé", "como se", "saber si",
                "menopausia o no", "es menopausia",
                "estoy en menopausia", "perimenopausia o no",
                "certeza", "hormonal o no",
                "sin tener todos los síntomas", "sin tener todos los sintomas"
            ])

            core_diagnosis_terms = [
                "menopause", "menopausal", "perimenopause",
                "menopausal transition", "menstrual",
                "amenorrhea", "amenorrhoea", "diagnosis", "clinical",
                "hot flashes", "hot flushes", "night sweats", "vasomotor",
                "vaginal dryness", "genitourinary", "final menstrual period"
            ]

            hard_diagnosis_terms = [
                "menstrual", "amenorrhea", "amenorrhoea", "irregular periods",
                "final menstrual period", "menopausal transition", "clinical diagnosis",
                "diagnosis", "hot flashes", "hot flushes", "night sweats",
                "vasomotor", "vaginal dryness", "genitourinary"
            ]

            mental_health_terms = [
                "mental health", "depression", "anxiety",
                "psychological", "psychiatric", "cognitive", "mood"
            ]

            if not has_any(full, core_diagnosis_terms):
                continue

            if diagnosis_query and not mental_query:
                strong_diagnostic_support = has_any(full, [
                    "clinical practice guideline", "clinical update",
                    "evaluation and management", "menopause and the perimenopause",
                    "menopause consultation", "clinical diagnosis",
                    "amenorrhea", "amenorrhoea", "amenorrheal",
                    "final menstrual period", "irregular periods",
                    "menstrual irregularity"
                ])
                if not strong_diagnostic_support:
                    continue
                if has_any(title, mental_health_terms):
                    continue
                if has_any(title, ["quality of life", "severity of menopausal symptoms", "relationship between"]):
                    if not has_any(full, ["clinical practice guideline", "clinical update", "evaluation and management", "amenorrhea", "amenorrhoea", "final menstrual period", "clinical diagnosis"]):
                        continue
                if has_any(full, mental_health_terms) and not has_any(full, hard_diagnosis_terms):
                    continue
                if has_any(title, ["aesthetic", "aesthetically", "skin", "hair", "alopecia"]):
                    continue
                if has_any(title, [
                    "physical activity", "exercise", "resistance training",
                    "hot flashes", "hot flushes", "vasomotor symptoms",
                    "duration of menopausal vasomotor symptoms",
                    "duration of vasomotor symptoms",
                    "reducing hot flushes", "reducing hot flashes",
                    "experiences and perceptions"
                ]):
                    continue

            if has_any(full, [
                "heart rate variability", "ventricular arrhythmia",
                "cardiac autonomic", "coronary heart disease",
                "cardiovascular risk"
            ]) and not has_any(q, [
                "corazón", "corazon", "cardiovascular",
                "palpitaciones", "heart"
            ]):
                continue

        elif question_type == "practical_guidance":
            if not has_any(full, [
                "guideline", "recommendation", "clinical",
                "management", "healthcare", "assessment",
                "screening", "diagnosis", "treatment",
                "follow-up", "menopause", "menopausal",
                "prevention", "preventive", "lifestyle", "exercise",
                "diet", "physical activity", "bone health",
                "cardiovascular risk"
            ]):
                continue

        elif question_type == "emotional_support":
            if not has_any(full, [
                "quality of life", "experience", "perceptions",
                "mental health", "depression", "anxiety",
                "psychological", "wellbeing", "symptoms",
                "menopause", "menopausal"
            ]):
                continue

        else:
            if question_intent == "risk" and has_any(q, ["cardiovascular", "corazón", "corazon", "heart"]):
                if has_any(full, ["eligibility criteria", "prescribing menopausal hormone therapy", "prescription of menopausal hormone therapy", "consensus document", "agreement document"]):
                    continue
                if not has_any(full, [
                    "cardiovascular risk", "cardiometabolic risk", "coronary heart disease",
                    "vascular health", "vascular aging", "atherosclerosis",
                    "blood pressure", "lipid", "metabolic", "risk factors",
                    "menopause transition"
                ]):
                    continue

            if question_intent == "cause" and has_any(q, ["calor", "sofoco", "sofocos", "vasomotor"]):
                if has_any(full, ["fezolinetant", "elinzanetant", "treatment", "therapy"]):
                    if not has_any(full, ["mechanism", "pathophysiology", "thermoregulatory", "hypothalamic", "neurokinin"]):
                        continue

            if question_intent == "treatment":
                if has_any(full, ["prevalence", "mapping global prevalence"]) and not has_any(full, [
                    "treatment", "therapy", "management", "intervention", "effective", "efficacy"
                ]):
                    continue

            if not has_any(full, [
                "menopause", "menopausal", "perimenopause",
                "postmenopausal", "vasomotor", "hot flashes",
                "hot flushes", "insomnia", "sleep", "depression", "anxiety",
                "cognitive", "hormone", "therapy", "treatment",
                "quality of life"
            ]):
                continue

        filtered.append(c)

    return filtered


def answer_rag(question, k=4, initial_k=25, max_per_doc=1, max_new_tokens=280, extra_terms=None):

    question_type = classify_question(question)

    question_intent = classify_intent(question)

    def context_has_any(terms, top_n=4):
        full = " ".join(
            ((c.get("title") or "") + " " + (c.get("text") or "")).lower()
            for c in contexts[:top_n]
        ) if "contexts" in locals() else ""
        return any(term in full for term in terms)

    def practical_fallback_answer(reason=None):
        q_l = question.lower()
        prefix = "El contexto científico recuperado no ofrece evidencia específica suficiente"
        if reason:
            prefix = f"{prefix} sobre {reason}"
        else:
            prefix = f"{prefix} para responder esta pregunta práctica"

        if question_intent == "monitoring":
            if any(t in q_l for t in ["qué información necesito", "que informacion necesito", "tomar decisiones", "organizar", "qué es importante", "que es importante", "visión global", "vision global", "mi estado"]):
                return (
                    f"{prefix}. Como orientación práctica de bajo riesgo, puede ser útil recopilar: síntomas principales, duración y frecuencia, patrón menstrual, impacto en sueño, ánimo, vida diaria o sexualidad, tratamientos o medicación, efectos adversos, antecedentes relevantes y dudas concretas. "
                    "No es una recomendación médica cerrada, sino una forma de ordenar la información para que la consulta sea más clara y para decidir con un profesional qué necesita valoración."
                )
            if any(t in q_l for t in ["qué seguimiento", "que seguimiento", "seguimiento", "revisiones", "controles", "seguimiento diario"]):
                return (
                    f"{prefix}. Como orientación práctica no diagnóstica, puede ser útil registrar patrón menstrual, sofocos o sudores, sueño, estado de ánimo, sequedad o síntomas genitourinarios, tratamientos o medicación, efectos adversos, antecedentes y grado de impacto. "
                    "Ese resumen puede ayudar en consulta a decidir qué requiere valoración clínica, qué puede observarse con prudencia y qué objetivos de seguimiento tienen sentido para tu caso."
                )
            if any(t in q_l for t in ["bien controlada", "controlada médicamente", "controlada medicamente", "médicamente", "medicamente"]):
                return (
                    f"{prefix}. No conviene inferir si una persona está bien controlada médicamente a partir de artículos sobre síntomas generales. "
                    "Para valorar control clínico, lo útil es revisar con un profesional la evolución de síntomas, impacto, tratamientos, efectos adversos, antecedentes, objetivos de salud y si existe un plan de seguimiento claro."
                )
            if any(t in q_l for t in ["analítica", "analitica", "análisis", "analisis", "prueba", "pruebas"]):
                return (
                    f"{prefix}. Por prudencia, no conviene convertir este contexto en una lista de analíticas concretas. "
                    "Para una consulta real, lo más útil sería llevar edad, patrón menstrual, síntomas, medicación, antecedentes y dudas, "
                    "y pedir que el profesional valore qué pruebas tienen sentido en tu caso."
                )
            if any(t in q_l for t in ["densitometría", "densitometria", "hueso", "ósea", "osea"]):
                return (
                    f"{prefix}. No puedo afirmar con estos documentos cuándo estaría indicada una densitometría o un control concreto. "
                    "La decisión depende de edad, antecedentes, fracturas, medicación y otros factores de riesgo, por lo que debe individualizarse en consulta."
                )
            if any(t in q_l for t in ["especialista", "ginecólogo", "ginecologo", "endocrino", "médico", "medico"]):
                return (
                    f"{prefix}. Como orientación no diagnóstica, puedes empezar por atención primaria o ginecología si la duda gira en torno a ciclo, sofocos, sangrado, sequedad vaginal o tratamiento. "
                    "Si predominan otras enfermedades o medicación compleja, el profesional inicial puede derivarte al especialista adecuado."
                )
            return (
                f"{prefix}. En vez de dar una recomendación clínica cerrada, conviene usar la consulta para ordenar síntomas, duración, impacto, medicación, antecedentes y prioridades de decisión. "
                "Eso reduce el riesgo de sacar conclusiones médicas de evidencia que no aborda tu caso directamente."
            )

        if question_intent == "prevention":
            return (
                f"{prefix}. Como orientación general de bajo riesgo, puede ser útil observar qué síntomas empeoran, cuidar sueño y estrés, mantener actividad física adaptada, revisar medicación y antecedentes, y pedir valoración si aparecen síntomas nuevos, intensos, persistentes o limitantes. "
                "No lo presentaría como prevención demostrada para tu caso: las decisiones preventivas concretas dependen de riesgo cardiovascular, salud ósea, antecedentes, hábitos y medicación, y deben individualizarse con un profesional."
            )

        if question_intent == "emotional":
            return (
                "El contexto recuperado no permite explicar tu caso concreto, pero tu preocupación no tiene por qué ser exagerada. "
                "Puede ayudarte registrar síntomas, momentos en que aparecen e impacto diario, y llevarlo a una valoración profesional si te está afectando."
            )

        return (
            f"{prefix}. Para no inventar recomendaciones, lo prudente es presentar esta pregunta como una duda clínica individual y llevar a consulta síntomas, duración, antecedentes, medicación actual y objetivos de la decisión."
        )

    def is_high_risk_request():
        q_l = question.lower()
        high_risk_terms = [
            "qué medicación", "que medicacion", "qué medicamento", "que medicamento",
            "qué tomar", "que tomar", "dosis", "cuánta dosis", "cuanta dosis",
            "recet", "prescrib", "tratamiento hormonal", "terapia hormonal",
            "hormonas debo", "debo tomar", "analítica", "analitica",
            "analíticas", "analiticas", "análisis", "analisis",
            "interpreta", "interpretar", "resultado", "resultados",
            "densitometría", "densitometria", "diagnostícame", "diagnosticame",
            "dime si tengo", "confirmar diagnóstico", "confirmar diagnostico"
        ]
        return any(t in q_l for t in high_risk_terms)

    def soft_guidance_fallback(reason=None):
        q_l = question.lower()

        if question_intent == "digital_orientation":
            q_l_words = re.findall(r"\b\w+\b", q_l)

            if is_high_risk_request():
                return (
                    "El contexto recuperado no permite usar una app o IA para diagnosticar, decidir medicación, ajustar dosis o sustituir una valoración médica. "
                    "Puede servir como apoyo para registrar información y preparar preguntas, pero las decisiones clínicas deben revisarse con un profesional."
                )

            mentions_app_or_ai = (
                any(t in q_l for t in ["app", "aplicación", "aplicacion", "inteligencia artificial"])
                or "ia" in q_l_words
            )

            if any(t in q_l for t in ["seguimiento diario", "registrar síntomas", "registrar sintomas"]) and not mentions_app_or_ai:
                return (
                    "Puede ser útil registrar patrón menstrual, sofocos o sudores, sueño, estado de ánimo, sequedad vaginal o síntomas genitourinarios, medicación, antecedentes e impacto en la vida diaria. "
                    "Ese registro puede ayudarte a ver patrones y preparar una consulta, pero no debe usarse para confirmar diagnósticos ni sustituir una valoración clínica."
                )

            return (
                "Una app puede servir como apoyo orientativo: registrar síntomas, ordenar información, preparar preguntas para la consulta y ofrecer educación general. "
                "No debe usarse para confirmar diagnósticos, decidir medicación o sustituir una valoración médica."
            )
        if question_intent == "monitoring":
            return (
                "Puede ser útil registrar patrón menstrual, sofocos o sudores, sueño, estado de ánimo, sequedad vaginal o síntomas genitourinarios, medicación, antecedentes e impacto en la vida diaria. "
                "Ese registro puede ayudar a preparar una consulta y a detectar cambios importantes, pero no sustituye la valoración clínica."
            )

        if question_intent == "prevention":
            return (
                "Como orientación general de bajo riesgo, puede ayudar observar qué empeora los síntomas, cuidar sueño y estrés, mantener actividad física adaptada y revisar medicación o antecedentes con un profesional si los síntomas son intensos o persistentes. "
                "No lo presentaría como prevención demostrada para tu caso ni como una recomendación clínica cerrada."
            )

        if question_type == "emotional_support" or question_intent == "emotional":
            return (
                "Lo que sientes puede ser real aunque no se pueda atribuir con certeza a la menopausia. "
                "Puede ayudar registrar cuándo aparece, qué lo empeora y qué impacto tiene, y comentarlo con un profesional si afecta a tu vida diaria."
            )

        if question_type == "limitation" and not is_high_risk_request():
            return (
                "El contexto recuperado no permite responder con una conclusión clínica cerrada. Aun así, como orientación general, puedes usar la información para ordenar síntomas, límites y dudas, sin convertirla en diagnóstico ni tratamiento individual."
            )

        return practical_fallback_answer(reason)

    def build_fallback_retrieval_query():
        q_l = question.lower()
        generic_terms = (
            " menopause menopausal symptoms climacteric symptoms perimenopause postmenopause "
            "vasomotor symptoms hot flashes night sweats sleep mood quality of life "
            "symptom tracking patient education clinical consultation health guidance"
        )
        practical_terms = (
            " symptom tracking menopause symptom diary patient education shared decision making "
            "clinical consultation quality of life menopause assessment health guidance "
            "self-monitoring patient reported outcomes Menopause Rating Scale MENQOL"
        )
        emotional_terms = (
            " menopause experience menopause perceptions quality of life emotional symptoms "
            "psychological symptoms mental health anxiety depression patient experience "
            "menopause transition supportive care"
        )
        digital_terms = (
            " digital health mobile app symptom tracking patient education self-monitoring "
            "clinical decision support menopause consultation health guidance limitations "
            "not diagnosis not medical advice"
        )

        extra = generic_terms
        if question_intent in ["monitoring", "digital_orientation"] or question_type == "practical_guidance":
            extra += practical_terms
        if question_type == "emotional_support" or question_intent == "emotional":
            extra += emotional_terms
        if question_intent == "digital_orientation":
            extra += digital_terms

        return question + extra

    def dynamic_min_score():
        if is_high_risk_request():
            return 0.45
        if question_intent == "digital_orientation":
            return 0.30
        if question_type == "emotional_support":
            return 0.30
        if question_type == "practical_guidance":
            return 0.35
        if question_type == "clinical_uncertainty":
            return 0.40
        return 0.45


    def evidence_overview():
        if not contexts:
            return "No hay contextos relevantes."
        type_counts = {}
        angles = []
        for c in contexts[:4]:
            c_type = c.get("type") or "unknown"
            type_counts[c_type] = type_counts.get(c_type, 0) + 1
            angle = c.get("evidence_angle") or "general"
            if angle not in angles:
                angles.append(angle)
        types_text = ", ".join(f"{v} {k}" for k, v in sorted(type_counts.items(), key=lambda x: -x[1]))
        angles_text = ", ".join(angles)
        return f"Fuentes recuperadas principales: {types_text}. Perspectivas cubiertas: {angles_text}."

    def controlled_synthesis_answer():
        q_l = question.lower()
        def refs_for_contexts(selected_contexts):
            pmids = []
            for c in selected_contexts:
                pmid = c.get("pmid")
                if pmid and str(pmid) not in pmids:
                    pmids.append(str(pmid))
            return ", ".join([f"PMID:{p}" for p in pmids])

        refs_text = refs_for_contexts(contexts[:3])
        suffix = f"\n\nReferencias: {refs_text}" if refs_text else ""

        if question_type == "clinical_uncertainty" and question_intent in ["diagnosis", "general"]:
            diagnostic_contexts = []
            diagnostic_terms = ["amenorrhea", "amenorrhoea", "amenorrheal", "menstrual", "final menstrual period", "clinical diagnosis", "irregular periods", "clinical practice guideline", "clinical update", "evaluation and management"]
            exclusion_title_terms = ["resistance training", "physical activity", "exercise", "hot flashes", "hot flushes", "vasomotor symptoms", "aesthetic", "aesthetically", "skin", "hair"]
            for c in contexts[:4]:
                title_l = (c.get("title") or "").lower()
                full_l = (title_l + " " + (c.get("text") or "")).lower()
                title_is_clinical = any(t in title_l for t in ["guideline", "clinical update", "evaluation and management", "menopause and the perimenopause"])
                hard_diagnostic = any(t in full_l for t in diagnostic_terms)
                weak_qol = any(t in title_l for t in ["quality of life", "severity of menopausal symptoms", "relationship between"])
                if (title_is_clinical or hard_diagnostic) and not weak_qol and not any(t in title_l for t in exclusion_title_terms):
                    diagnostic_contexts.append(c)
            refs_text = refs_for_contexts(diagnostic_contexts)
            suffix = f"\n\nReferencias: {refs_text}" if refs_text else ""
            if any(t in q_l for t in ["diferencio", "diferenciar", "hormonal o no", "hormonales o no"]):
                answer = (
                    "No se puede distinguir con certeza si un síntoma es hormonal solo por cómo se siente. Para orientarlo, suele ayudar mirar el contexto: edad, cambios del ciclo, irregularidad menstrual, aparición junto a sofocos o sudores nocturnos, sequedad vaginal, sueño y evolución temporal. "
                    "También hay síntomas que pueden tener causas no hormonales, como estrés, medicación u otros problemas de salud. Si el síntoma es nuevo, intenso, persistente o no encaja con tu patrón habitual, lo más prudente es registrarlo y revisarlo en consulta."
                )
            elif any(t in q_l for t in ["certeza", "qué cosas son", "que cosas son", "cuáles no", "cuales no"]):
                answer = (
                    "No suele existir una forma perfecta de atribuir cada síntoma a la menopausia y descartar todo lo demás solo con una pregunta o una lista de síntomas. La orientación se apoya en la edad, la historia menstrual, la irregularidad del ciclo y, en menopausia establecida, la ausencia de regla durante 12 meses. "
                    "Los sofocos, sudores nocturnos o sequedad vaginal pueden acompañar la transición, pero no explican por sí solos todos los cambios. Un registro de síntomas y una valoración clínica ayudan a separar lo probable, lo dudoso y lo que conviene estudiar por otra vía."
                )
            elif any(t in q_l for t in ["sin tener todos", "no tenga todos", "no tengo todos", "todos los síntomas", "todos los sintomas"]):
                answer = (
                    "Sí, es posible estar en transición menopáusica o llegar a la menopausia sin tener todos los síntomas típicos. Los síntomas varían mucho entre mujeres: algunas tienen sofocos intensos, otras cambios del sueño o sequedad vaginal, y otras pocos síntomas. "
                    "Para orientarse pesan más la edad, la historia menstrual, la irregularidad del ciclo y, en menopausia establecida, la ausencia de regla durante 12 meses que tener una lista completa de síntomas. Si hay dudas o cambios llamativos, conviene revisarlo con un profesional."
                )
            elif any(t in q_l for t in ["todo lo que me pasa", "relacionando mal", "por otra cosa", "solo estrés", "solo estres"]):
                answer = (
                    "No conviene asumir que todo lo que aparece en esta etapa se debe a la menopausia. Algunos cambios pueden estar relacionados con la transición hormonal, como irregularidad menstrual, sofocos, sudores nocturnos o sequedad vaginal, pero otros pueden depender de estrés, sueño, medicación, antecedentes u otras causas. "
                    "La forma más segura de orientarlo es registrar qué ocurre, desde cuándo, con qué frecuencia y qué impacto tiene, y revisar los síntomas nuevos, intensos o persistentes con un profesional sanitario."
                )
            else:
                answer = (
                    "No se puede confirmar solo por síntomas si lo que te ocurre es menopausia, perimenopausia u otra causa. "
                    "Como orientación, tienen especial peso la edad, la historia menstrual, la irregularidad del ciclo y, en menopausia establecida, la ausencia de regla durante 12 meses; también pueden acompañar sofocos, sudores nocturnos o sequedad vaginal. "
                    "Si los cambios son nuevos, intensos, persistentes o no encajan con tu patrón habitual, conviene registrarlos y revisarlos con un profesional sanitario para descartar otras causas."
                )
            return answer + suffix

        if question_intent == "treatment" and any(t in q_l for t in ["sofoco", "sofocos", "calor", "hot flash", "hot flush", "vasomotor"]):
            full = " ".join(((c.get("title") or "") + " " + (c.get("text") or "")).lower() for c in contexts[:4])
            options = []
            if any(t in full for t in ["hormone therapy", "menopausal hormone therapy", "estrogen therapy", "hormonal"]):
                options.append("terapia hormonal cuando está indicada y no hay contraindicaciones")
            if any(t in full for t in ["fezolinetant", "elinzanetant", "non-hormonal", "nonhormonal"]):
                options.append("tratamientos no hormonales, especialmente fármacos dirigidos a síntomas vasomotores como fezolinetant si procede clínicamente")
            if any(t in full for t in ["mindfulness", "cbt", "cognitive behavioral", "exercise", "acupuncture", "lifestyle"]):
                options.append("intervenciones no farmacológicas evaluadas en los estudios, como mindfulness, enfoques cognitivo-conductuales, ejercicio u otras estrategias de apoyo")
            if not options:
                return None
            answer = (
                "Los contextos recuperados no apuntan a una única opción universal para los sofocos, sino a varias líneas de manejo. "
                "Entre las opciones descritas están " + "; ".join(options) + ". "
                "La elección depende de intensidad de los síntomas, antecedentes, preferencias y contraindicaciones, por lo que no debe interpretarse como una prescripción individual."
            )
            return answer + suffix

        if (question_intent == "treatment" or any(t in q_l for t in ["cbt", "cbt-i", "tcc", "terapia cognitivo"])) and any(t in q_l for t in ["insomnio", "dormir", "sueño", "sleep", "insomnia", "calidad del sueño"]):
            full = " ".join(((c.get("title") or "") + " " + (c.get("text") or "")).lower() for c in contexts[:4])
            if any(t in full for t in ["cbt-i", "cbti", "cognitive behavioral therapy", "cognitive behavioural therapy"]):
                answer = (
                    "Una de las opciones con mejor evidencia recuperada para el insomnio relacionado con la menopausia es la terapia cognitivo-conductual para el insomnio (CBT-I). "
                    "Los meta-análisis recuperados sugieren mejoras en la calidad del sueño y en la gravedad del insomnio, aunque no debe presentarse como la única opción ni como una indicación individual. "
                    "La elección del manejo depende de la causa del insomnio, síntomas asociados, preferencias y valoración profesional."
                )
                return answer + suffix

        if question_intent == "monitoring" and any(t in q_l for t in ["visión global", "vision global", "qué es importante", "que es importante", "guía clara", "guia clara", "seguimiento diario", "qué seguimiento", "que seguimiento", "seguimiento", "mi estado"]):
            answer = (
                "El contexto biomédico recuperado no ofrece una herramienta concreta para tener una visión global diaria. "
                "Como orientación práctica no diagnóstica, puede ser útil ordenar la información por patrón menstrual, sofocos o sudores, sueño, estado de ánimo, sequedad o síntomas genitourinarios, medicación, antecedentes y grado de impacto en la vida diaria. "
                "Lo importante sería llevar ese resumen a consulta para decidir qué requiere seguimiento clínico y qué puede observarse con prudencia."
            )
            return answer + suffix

        return None

    retrieval_mode = "normal"

    retrieval_query = expand_query_for_retrieval(question)

    if extra_terms:
        retrieval_query = retrieval_query + " " + extra_terms

    contexts = retrieve(
        retrieval_query,
        k=k,
        initial_k=initial_k,
        max_per_doc=max_per_doc,
        use_type_priority=True
    )

    contexts = filter_contexts(
        question,
        contexts,
        question_type=question_type,
        min_score=dynamic_min_score(),
        question_intent=question_intent
    )

    provisional_top_score = 0.0
    if contexts:
        provisional_top_score = contexts[0].get("adjusted_score", contexts[0].get("score", 0.0))

    if not contexts or provisional_top_score == 0.0:
        fallback_retrieval_query = build_fallback_retrieval_query()
        fallback_contexts = retrieve(
            fallback_retrieval_query,
            k=k,
            initial_k=max(initial_k, 50),
            max_per_doc=max_per_doc
        )
        fallback_contexts = filter_contexts(
            question,
            fallback_contexts,
            question_type=question_type,
            min_score=dynamic_min_score(),
            question_intent=question_intent
        )
        fallback_top_score = 0.0
        if fallback_contexts:
            fallback_top_score = fallback_contexts[0].get("adjusted_score", fallback_contexts[0].get("score", 0.0))
        if fallback_contexts and fallback_top_score >= provisional_top_score:
            contexts = fallback_contexts
            retrieval_query = fallback_retrieval_query
            retrieval_mode = "fallback_expansion"

    # PMIDs RECUPERADOS

    used_pmids = []

    for c in contexts[:3]:
        pmid = c.get("pmid")

        if pmid and pmid not in used_pmids:
            used_pmids.append(str(pmid))

    # SCORE MÁXIMO

    top_score = 0.0

    if contexts:
        top_score = contexts[0].get(
            "adjusted_score",
            contexts[0].get("score", 0.0)
        )

    unsupported_practical_topics = [
        {
            "reason": "analíticas o pruebas hormonales",
            "triggers": ["analítica", "analitica", "analíticas", "analiticas", "análisis", "analisis", "pruebas hormonales", "estrógeno", "estrogeno", "progesterona", "fsh", "tiroides"],
            "support": ["blood test", "laboratory", "serum", "fsh", "follicle-stimulating", "estradiol level", "oestrogen level", "estrogen level", "thyroid", "diagnostic test"]
        },
        {
            "reason": "vitaminas o suplementos",
            "triggers": ["vitamina", "vitaminas", "suplemento", "suplementos", "magnesio", "omega", "calcio", "vitamina d"],
            "support": ["vitamin", "supplement", "calcium", "vitamin d", "magnesium", "omega"]
        },
        {
            "reason": "densitometría o controles óseos",
            "triggers": ["densitometría", "densitometria", "hueso", "ósea", "osea", "osteoporosis"],
            "support": ["bone mineral density", "densitometry", "dxa", "osteoporosis", "fracture risk"]
        },
        {
            "reason": "control médico o seguimiento clínico",
            "triggers": ["bien controlada", "controlada médicamente", "controlada medicamente", "médicamente", "medicamente"],
            "support": ["follow-up", "clinical follow-up", "monitoring", "management plan", "clinical practice guideline", "evaluation and management", "menopause consultation"]
        }
    ]

    if contexts and question_type == "practical_guidance":
        q_l = question.lower()
        for topic in unsupported_practical_topics:
            if any(t in q_l for t in topic["triggers"]) and not context_has_any(topic["support"]):
                answer = practical_fallback_answer(topic["reason"])
                return {
                    "question": question,
                    "question_type": question_type,
                    "question_intent": question_intent,
                    "retrieval_query": retrieval_query,
                    "answer": answer,
                    "contexts": contexts,
                    "used_pmids": used_pmids,
                    "top_score": top_score,
                    "retrieval_mode": retrieval_mode,
                    "prompt": None
                }

    if question_type == "practical_guidance" and question_intent == "prevention":
        answer = soft_guidance_fallback()
        retrieval_mode = "soft_fallback"
        return {
            "question": question,
            "question_type": question_type,
            "question_intent": question_intent,
            "retrieval_query": retrieval_query,
            "answer": answer,
            "contexts": contexts,
            "used_pmids": used_pmids,
            "top_score": top_score,
            "retrieval_mode": retrieval_mode,
                    "prompt": None
        }

    controlled_answer = controlled_synthesis_answer()
    if controlled_answer:
        return {
            "question": question,
            "question_type": question_type,
            "question_intent": question_intent,
            "retrieval_query": retrieval_query,
            "answer": controlled_answer,
            "contexts": contexts,
            "used_pmids": used_pmids,
            "top_score": top_score,
            "retrieval_mode": retrieval_mode,
                    "prompt": None
        }

    MIN_RELEVANCE_SCORE = dynamic_min_score()

    # SI NO HAY CONTEXTO RELEVANTE

    if not contexts or top_score < MIN_RELEVANCE_SCORE:

        if question_type == "practical_guidance":
            if is_high_risk_request():
                answer = practical_fallback_answer()
                retrieval_mode = "hard_fallback"
            else:
                answer = soft_guidance_fallback()
                retrieval_mode = "soft_fallback"

        elif question_type == "emotional_support":
            answer = soft_guidance_fallback()
            retrieval_mode = "soft_fallback"

        elif question_type == "clinical_uncertainty":
            answer = (
                "No se puede saber con seguridad solo a partir de esta información si lo que te ocurre está relacionado con "
                "la menopausia, la perimenopausia u otra causa. El contexto científico recuperado no permite responder esta "
                "pregunta de forma específica y fiable. Si los síntomas son nuevos, intensos, persistentes o te preocupan, "
                "lo más recomendable es consultarlo con un profesional sanitario y registrar qué notas, desde cuándo y con "
                "qué frecuencia aparece."
            )

        else:
            answer = (
                "No hay suficiente información relevante "
                "en el contexto científico recuperado."
            )

        if used_pmids:
            refs_text = ", ".join([f"PMID:{p}" for p in used_pmids])
            answer = f"{answer}\n\nReferencias: {refs_text}"

        return {
            "question": question,
            "question_type": question_type,
            "retrieval_query": retrieval_query,
            "answer": answer,
            "contexts": contexts,
            "used_pmids": used_pmids,
            "top_score": top_score,
            "question_intent": question_intent,
            "retrieval_mode": retrieval_mode,
                    "prompt": None
        }

    # CONTEXTO

    context_text = build_context(
        contexts,
        max_chars=3000,
        max_per_doc=max_per_doc,
        question=question,
        question_intent=question_intent
    )

    evidence_summary = evidence_overview()

    intent_rules = {
        "cause": "La pregunta pide causa o mecanismo: responde primero por qué ocurre; no conviertas la respuesta en una lista de tratamientos salvo que el contexto lo conecte directamente.",
        "treatment": "La pregunta pide qué hacer o tratamiento: prioriza opciones de manejo apoyadas por el contexto; no sustituyas esto por prevalencia o asociaciones generales.",
        "risk": "La pregunta pide riesgo cardiovascular: responde sobre riesgo, transición menopáusica y factores vasculares; no conviertas la respuesta en criterios de terapia hormonal si eso no se pregunta.",
        "diagnosis": "La pregunta pide orientación diagnóstica: prioriza ciclo menstrual, amenorrea, transición menopáusica y síntomas vasomotores/genitourinarios; no centres la respuesta en salud mental si la usuaria no pregunta por ánimo.",
        "prognosis": "La pregunta pide duración o evolución: responde sobre curso temporal y límites de la evidencia.",
        "monitoring": "La pregunta pide pruebas, controles, especialista o seguimiento: no enumeres analíticas, vitaminas ni pruebas concretas si no aparecen de forma explícita en el contexto.",
        "prevention": "La pregunta pide prevención o estilo de vida: distingue evidencia recuperada de consejos preventivos generales y reconoce si el corpus no cubre esa dimensión.",
        "emotional": "La pregunta pide apoyo: valida de forma prudente y evita sobrediagnosticar.",
        "practical": "La pregunta pide orientación sanitaria práctica: distingue lo que dice el contexto de lo que requiere consulta individual.",
        "support": "La pregunta pide apoyo: valida de forma prudente y evita sobrediagnosticar.",
        "general": "Responde a la intención principal de la pregunta sin añadir temas tangenciales."
    }
    intent_instruction = intent_rules.get(question_intent, intent_rules["general"])

    # =========================================================
    # PROMPT CLINICAL UNCERTAINTY
    # =========================================================

    if question_type == "clinical_uncertainty":

        prompt = f"""
        Eres un asistente biomédico especializado en menopausia, perimenopausia y salud hormonal femenina.

        La usuaria está intentando interpretar síntomas o entender si lo que experimenta podría estar relacionado con la menopausia.

        Usa únicamente el contexto científico proporcionado, pero responde de forma prudente, clara y orientativa.

        Instrucciones:
        - {intent_instruction}
        - Responde directamente a la duda principal de la usuaria.
        - Explica qué síntomas o cambios suelen asociarse con la menopausia o la transición menopáusica si aparecen en el contexto.
        - En preguntas diagnósticas generales, prioriza cambios del ciclo menstrual, amenorrea, sofocos, sudores nocturnos, sequedad vaginal y límites diagnósticos.
        - No centres la respuesta en ansiedad, depresión o cognición si la pregunta no menciona ánimo, salud mental o memoria.
        - No afirmes que la usuaria tenga una condición concreta.
        - No hagas diagnósticos médicos.
        - No fuerces conclusiones si la evidencia recuperada es indirecta.
        - Si el contexto es limitado, dilo claramente.
        - Prioriza información clínica útil frente a prevalencia o asociaciones generales.
        - Recomienda consultar con un profesional sanitario cuando la situación requiera valoración individual.
        - No repitas la pregunta.
        - No empieces con "Basándose en el contexto".

        Tipo de pregunta:
        {question_type}

        Intención de la pregunta:
        {question_intent}

        Responde en español en 3-5 frases naturales y prudentes.

        Pregunta:
        {question}

        Resumen de evidencia recuperada:
        {evidence_summary}

        Contexto científico:
        {context_text}

        Respuesta:
        """

    # =========================================================
    # PROMPT EMOTIONAL SUPPORT
    # =========================================================

    elif question_type == "emotional_support":

        prompt = f"""
        Eres un asistente especializado en acompañamiento informativo sobre menopausia y salud hormonal femenina.

        La usuaria expresa preocupación, inseguridad o necesidad de orientación emocional relacionada con la menopausia.

        Instrucciones:
        - {intent_instruction}
        - Responde de forma empática pero profesional.
        - No infantilices ni exageres.
        - No hagas diagnósticos médicos.
        - No inventes evidencia científica.
        - Puedes explicar de forma general que muchas mujeres experimentan incertidumbre o cambios importantes durante esta etapa.
        - Si el contexto científico recuperado es limitado, no fuerces detalles biomédicos irrelevantes.
        - Prioriza claridad, orientación y validación prudente.
        - No repitas la pregunta.

        Responde en español en 3-5 frases naturales.

        Pregunta:
        {question}

        Resumen de evidencia recuperada:
        {evidence_summary}

        Contexto científico:
        {context_text}

        Respuesta:
        """

    # =========================================================
    # PROMPT PRACTICAL GUIDANCE
    # =========================================================

    elif question_type == "practical_guidance":

        prompt = f"""
        Eres un asistente biomédico especializado en menopausia y salud hormonal femenina.

        La usuaria busca orientación práctica o de navegación sanitaria.

        Instrucciones:
        - {intent_instruction}
        - Responde de forma clara y útil.
        - Prioriza información práctica frente a prevalencia o asociaciones generales.
        - No hagas diagnósticos médicos.
        - No inventes recomendaciones clínicas no presentes en el contexto.
        - Si la pregunta pide analíticas, vitaminas, densitometría o pruebas y el contexto no las menciona explícitamente, di que el contexto no permite concretarlas.
        - Si el contexto es limitado, dilo claramente.
        - Puedes recomendar consultar con profesionales sanitarios apropiados cuando corresponda.
        - No repitas la pregunta.

        Responde en español en 3-5 frases claras.

        Pregunta:
        {question}

        Resumen de evidencia recuperada:
        {evidence_summary}

        Contexto científico:
        {context_text}

        Respuesta:
        """

    # PROMPT LIMITACIÓN

    elif question_type == "limitation":

        answer = (
            "No hay suficiente evidencia específica en el contexto "
            "científico recuperado para responder esta pregunta "
            "de forma fiable."
        )

        return {
            "question": question,
            "question_type": question_type,
            "retrieval_query": retrieval_query,
            "answer": answer,
            "contexts": contexts,
            "used_pmids": used_pmids,
            "top_score": top_score,
            "question_intent": question_intent,
            "retrieval_mode": retrieval_mode,
                    "prompt": None
        }

    # PROMPT CIENTÍFICO

    else:

        prompt = f"""
          Eres un asistente de investigación biomédica especializado en menopausia, perimenopausia y salud hormonal femenina.

          Debes responder usando únicamente el contexto científico proporcionado.

          Instrucciones:
        - {intent_instruction}
          - Responde directamente a la pregunta del usuario.
          - Sintetiza la evidencia más relevante de forma clara y útil.
          - Prioriza información clínica o práctica frente a asociaciones generales o prevalencia.
          - No hagas diagnósticos médicos individuales.
          - No inventes datos ni relaciones causales no apoyadas por el contexto.
          - Si la evidencia es limitada o indirecta, indícalo de forma breve y prudente.
          - Si el contexto no responde claramente a la pregunta, evita forzar detalles tangenciales.
          - No repitas la pregunta.
          - No uses frases como "basándose en el contexto".
          - Identifica primero la intención de la pregunta:
            - si pregunta "por qué", responde primero con la causa o mecanismo;
            - si pregunta "qué puedo hacer" o "tratamiento", responde con opciones de manejo;
            - si pregunta "cómo sé", responde con criterios orientativos y límites diagnósticos;
            - si pregunta "cuánto dura" o "para siempre", responde sobre duración o evolución.
          - No respondas con tratamientos si la pregunta pide una causa.
          - No respondas con prevalencia si la pregunta pide qué hacer.
          - Si la intención es tratamiento, usa al menos dos perspectivas recuperadas cuando existan; no limites la respuesta al primer PMID.
          - Si la intención es riesgo cardiovascular, evita hablar de prescripción de terapia hormonal salvo que la pregunta lo pida.
          Regla de intención concreta: {intent_instruction}

          Responde en español en 3-5 frases claras y naturales.

          Tipo de pregunta:
          {question_type}

          Intención de la pregunta:
          {question_intent}

          Pregunta:
          {question}

          Resumen de evidencia recuperada:
          {evidence_summary}

          Contexto científico:
          {context_text}

          Respuesta:
          """

    # GENERACIÓN

    answer = generate_answer(
        prompt,
        max_new_tokens=max_new_tokens
    )

    answer = clean_answer(answer)

    # REFERENCIAS

    if used_pmids:

        refs_text = ", ".join([
            f"PMID:{p}" for p in used_pmids
        ])

        answer = f"{answer}\n\nReferencias: {refs_text}"

    return {
        "question": question,
        "question_type": question_type,
        "question_intent": question_intent,
        "retrieval_query": retrieval_query,
        "answer": answer,
        "contexts": contexts,
        "used_pmids": used_pmids,
        "top_score": top_score,
        "retrieval_mode": retrieval_mode,
        "prompt": prompt
    }

# Quick domain test

# Lista de preguntas de ejemplo
questions = [
    # Español
    "¿Cómo sé si estos despistes son normales o si deberían preocuparme?",
    "¿Cómo sé si me falta alguna vitamina?",
    "¿Cómo puedo saber si estoy bien controlada médicamente?",
    "¿Cómo puedo prevenir problemas futuros?",
    "¿Es tarde para empezar a prevenir?",
    "¿La terapia cognitivo-conductual para el insomnio mejora la calidad del sueño y la gravedad del insomnio en mujeres menopáusicas?",
    "¿Qué puedo hacer con los sofocos durante la menopausia?",
    "¿Cómo afecta la menopausia a la salud mental?",
    "¿La menopausia es igual para todas las mujeres?",
    "¿El estrés puede empeorar los síntomas de la menopausia?",
    "¿Cómo afectan los cambios hormonales a la calidad del sueño en mujeres postmenopáusicas?",
    "¿Cuál es el mejor tratamiento para el insomnio durante la menopausia?",
    "¿La menopausia aumenta el riesgo de enfermedad cardiovascular?",
    "¿Puede el mindfulness ayudar con los síntomas de la menopausia?",
    "¿Es eficaz la terapia cognitivo-conductual en mujeres menopáusicas?",

    # Inglés
    "Does CBT-I improve sleep quality and insomnia severity in menopausal women?",
    "Can stress contribute to menopause symptoms?"
]

# Ejecutamos el modelo para cada pregunta
for q in questions:
    rag_result = answer_rag(
        q,
        k=5,
        initial_k=15,
        max_per_doc=1,
        max_new_tokens=220
    )

    print("\n" + "="*80)
    print("QUESTION:\n", rag_result["question"])
    print("\nQUESTION TYPE:\n", rag_result["question_type"])
    if "question_intent" in rag_result:
        print("\nQUESTION INTENT:\n", rag_result["question_intent"])
    print("\nTOP SCORE:\n", rag_result["top_score"])

    if "retrieval_query" in rag_result:
        print("\nRETRIEVAL QUERY:\n", rag_result["retrieval_query"])

    if "retrieval_mode" in rag_result:
        print("\nRETRIEVAL MODE:\n", rag_result["retrieval_mode"])

    print("\nANSWER:\n", rag_result["answer"])

    print("\nTOP CONTEXTS:")
    for c in rag_result["contexts"][:4]:
        print(f"- PMID={c['pmid']} | type={c['type']} | title={c['title']}")



QUESTION:
 ¿Cómo sé si estos despistes son normales o si deberían preocuparme?

QUESTION TYPE:
 clinical_uncertainty

QUESTION INTENT:
 diagnosis

TOP SCORE:
 1.125657693862915

RETRIEVAL QUERY:
 ¿Cómo sé si estos despistes son normales o si deberían preocuparme? menopause diagnosis clinical diagnosis menopausal transition perimenopause 12 months amenorrhea amenorrhoea final menstrual period menstrual irregularity irregular periods vasomotor symptoms hot flashes hot flushes night sweats vaginal dryness genitourinary symptoms menopause cognition cognitive function memory brain fog concentration executive function cognitive complaints perimenopause postmenopause

RETRIEVAL MODE:
 normal

ANSWER:
 No se puede confirmar solo por síntomas si lo que te ocurre es menopausia, perimenopausia u otra causa. Como orientación, tienen especial peso la edad, la historia menstrual, la irregularidad del ciclo y, en menopausia establecida, la ausencia de regla durante 12 meses; también pueden acompañar

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



QUESTION:
 ¿Cómo afecta la menopausia a la salud mental?

QUESTION TYPE:
 scientific

QUESTION INTENT:
 general

TOP SCORE:
 0.8577119064331055

RETRIEVAL QUERY:
 ¿Cómo afecta la menopausia a la salud mental? menopause mental health depression anxiety stress psychological symptoms mood irritability depressive symptoms postmenopausal women perimenopausal women quality of life mindfulness CBT mind-body therapies menopause menopausal symptoms climacteric symptoms quality of life Menopause Rating Scale MRS MENQOL vasomotor psychological physical sexual symptoms perimenopausal women postmenopausal women

RETRIEVAL MODE:
 normal

ANSWER:
 La menopausia puede afectar significativamente la salud mental de las mujeres, aumentando el riesgo de depresión, ansiedad e insomnio. Según una revisión sistemática y meta-análisis, la prevalencia de síntomas depresivos fue del 32% en mujeres perimenopáusicas y del 30% en mujeres posmenopáusicas. Además, se encontró que casi una de cada tres mujeres exper

In [ ]:
# =========================
# Prueba controlada de términos extra
# =========================

test_question = "¿Cómo sé si lo que me está pasando es menopausia o no?"

terms_prueba_1 = (
    "menopause symptoms "
    "menopausal symptoms "
    "menopause diagnosis "
    "women health during menopause"
)

terms_prueba_2 = (
    "menopause symptoms "
    "menopausal symptoms "
    "menopause diagnosis "
    "menopausal transition "
    "perimenopause symptoms "
    "women health during menopause"
)


In [ ]:
for name, terms in [
    ("PRUEBA 1", terms_prueba_1),
    ("PRUEBA 2", terms_prueba_2)
]:
    result = answer_rag(
        test_question,
        k=4,
        initial_k=25,
        max_per_doc=1,
        max_new_tokens=280,
        extra_terms=terms
    )

    print("\n" + "="*80)
    print(name)
    print("="*80)

    print("\nQUESTION:")
    print(result["question"])

    print("\nRETRIEVAL QUERY:")
    print(result["retrieval_query"])

    print("\nTOP SCORE:")
    print(result["top_score"])

    print("\nRAG ANSWER:")
    print(result["answer"])

    print("\nTOP CONTEXTS:")
    for c in result["contexts"][:4]:
        print(f"- PMID={c['pmid']} | type={c['type']} | title={c['title']}")



PRUEBA 1

QUESTION:
¿Cómo sé si lo que me está pasando es menopausia o no?

RETRIEVAL QUERY:
¿Cómo sé si lo que me está pasando es menopausia o no? menopause diagnosis clinical diagnosis menopausal transition perimenopause 12 months amenorrhea amenorrhoea final menstrual period menstrual irregularity irregular periods vasomotor symptoms hot flashes hot flushes night sweats vaginal dryness genitourinary symptoms menopause symptoms menopausal symptoms menopause diagnosis women health during menopause

TOP SCORE:
1.126651749610901

RAG ANSWER:
No se puede confirmar solo por síntomas si lo que te ocurre es menopausia, perimenopausia u otra causa. Como orientación, tienen especial peso la edad, la historia menstrual, la irregularidad del ciclo y, en menopausia establecida, la ausencia de regla durante 12 meses; también pueden acompañar sofocos, sudores nocturnos o sequedad vaginal. Si los cambios son nuevos, intensos, persistentes o no encajan con tu patrón habitual, conviene registrarlos 

Test retrieval 2

In [ ]:
test_queries = [
    "¿Qué puedo hacer con los sofocos durante la menopausia?",
    "¿Puede el mindfulness ayudar con los síntomas de la menopausia?",
    "¿La menopausia aumenta el riesgo de enfermedad cardiovascular?",
    "¿La terapia cognitivo-conductual para el insomnio mejora la calidad del sueño?",
    "Can stress contribute to menopause symptoms?"
]

for q in test_queries:
    expanded = expand_query_for_retrieval(q)

    res = retrieve(
        expanded,
        k=5,
        initial_k=40,
        max_per_doc=1,
        use_type_priority=True,
        use_keyword_rerank=True
    )

    print("\n" + "="*80)
    print("QUESTION:", q)
    print("EXPANDED:", expanded[:300], "...")

    for r in res:
        print(
            f"adj={r['adjusted_score']:.3f} | "
            f"base={r['score']:.3f} | "
            f"kw={r['keyword_score']:.3f} | "
            f"pen={r['noise_penalty']:.3f} | "
            f"type={r['type']} | PMID={r['pmid']}"
        )
        print("title:", r["title"])
        print("---")



QUESTION: ¿Qué puedo hacer con los sofocos durante la menopausia?
EXPANDED: ¿Qué puedo hacer con los sofocos durante la menopausia? hot flashes hot flushes vasomotor symptoms night sweats menopause treatment management hormone therapy nonhormonal therapy fezolinetant elinzanetant CBT mindfulness exercise acupuncture ...
adj=1.085 | base=0.657 | kw=0.308 | pen=0.000 | type=meta-analysis | PMID=41930718
title: Fezolinetant treatment for hot flashes associated with menopause: A plain language summary of the DAYLIGHT study.
---
adj=1.027 | base=0.611 | kw=0.296 | pen=0.000 | type=meta-analysis | PMID=38583073
title: Menopausal symptom management: Fezolinetant's varied doses provide effective relief for vasomotor symptoms in women - A meta-analysis of 3291 participants.
---
adj=0.921 | base=0.655 | kw=0.166 | pen=0.000 | type=included-study-candidate | PMID=16952511
title: A randomized controlled pilot study of acupuncture for postmenopausal hot flashes: effect on nocturnal hot flashes and

12) Baseline vs RAG comparison

In [ ]:
# =========================
# 12) Baseline vs RAG comparison
# =========================

def answer_base(question, max_new_tokens=220):
    prompt = f"""You are a helpful biomedical assistant specializing in female hormonal health,
    menstrual cycle, fertility, perimenopause, menopause, postmenopause, thyroid health, and metabolic changes.
    You are also an expert in symptoms such as sleep disturbances, stress, and other hormonal changes throughout a woman's life.

    Answer the question clearly in 2-4 sentences.
    If you are not sure, say so.

    Question:
    {question}

    Answer:
    """
    answer = generate_answer(prompt, max_new_tokens=max_new_tokens)
    answer = clean_answer(answer)
    return answer

queries = {
    "Incertidumbre diagnóstica": [
        "¿Cómo sé si lo que me está pasando es menopausia o no?",
        "¿Esto que me pasa es por la edad o por otra cosa?",
        "¿Todo lo que me pasa ahora tiene que ver con la menopausia?",
        "¿Cómo diferencio si un síntoma es hormonal o no?",
        "¿Puede ser solo estrés y no menopausia?",
        "¿Estoy en menopausia aunque no tenga todos los síntomas?",
        "¿Esto es normal en esta etapa o debería preocuparme?",
        "¿Hay alguna forma de saber con certeza qué cosas son de la menopausia y cuáles no?",
        "¿Todo lo que me pasa ahora lo estoy relacionando mal con la menopausia?",
        "¿Estoy exagerando o realmente me está pasando algo?"
    ],

    "Necesidad de orientación personalizada": [
        "¿Qué pruebas médicas debería hacerme ahora mismo?",
        "¿Qué tipo de especialista debería consultar?",
        "¿Qué seguimiento debería llevar en esta etapa?",
        "¿Qué analíticas son importantes en esta etapa de la menopausia?",
        "¿Cómo sé si me falta alguna vitamina?",
        "¿Qué debería revisar en mis análisis de sangre?",
        "¿Qué controles debería hacerme de forma preventiva?",
        "¿Cuándo debería hacerme una densitometría?",
        "¿Qué revisiones son realmente necesarias y cuáles no?",
        "¿Cómo puedo saber si estoy bien controlada médicamente?"
    ],

    "Demanda de acompañamiento": [
        "¿Por qué nadie me explica claramente qué me está pasando?",
        "¿Existe alguna forma de tener seguimiento diario de mis síntomas?",
        "¿Cómo sería tener un acompañamiento diario en esta etapa?",
        "¿Qué información necesito para tomar decisiones sobre mi salud?",
        "¿Cómo puedo tener una guía clara de qué hacer en cada momento?",
        "¿Por qué siento que voy a ciegas en este proceso?",
        "¿Cómo puedo organizar toda la información que recibo?",
        "¿Cómo puedo saber qué es importante y qué no?",
        "¿Cómo puedo tener una visión global de mi estado?"
    ],

    "Prevención": [
        "¿Qué debería haber hecho antes para prevenir esto?",
        "¿A qué edad debería haber empezado a cuidarme?",
        "¿Qué puedo hacer ahora para evitar empeorar?",
        "¿Es tarde para empezar a prevenir?",
        "¿Qué hábitos son clave en esta etapa?",
        "¿Cómo puedo prevenir problemas futuros?",
        "¿Qué debería cambiar en mi estilo de vida?",
        "¿Qué cosas empeoran los síntomas sin darme cuenta?",
        "¿Cómo influye la alimentación en esta etapa?",
        "¿Qué debería haber sabido antes?"
    ],

    "Cuerpo y cambios físicos": [
        "¿Por qué me están pasando cosas en el cuerpo que nunca me habían pasado?",
        "¿Es normal que cambie la piel de repente?",
        "¿Por qué tengo picores o cambios en la piel?",
        "¿Es normal tener más sequedad corporal?",
        "¿Por qué mi cuerpo reacciona diferente ahora?",
        "¿Esto que me pasa en el cuerpo es temporal?",
        "¿Mi cuerpo va a volver a ser como antes?",
        "¿Estos cambios son reversibles?",
        "¿Cuánto tiempo duran estos cambios físicos?",
        "¿Qué cambios son normales y cuáles no?",
        "¿De verdad estos cambios físicos dependen tanto de la genética?",
        "¿Me voy a parecer más a mi madre y a mi abuela materna que a la familia de mi padre en cómo llevo la menopausia?",
        "¿Por qué me engorda la barriga más que todo lo demás?",
        "¿Me puedo quedar medio calva o sin pelo por la menopausia?",
        "¿Por qué parece que las famosas no tienen menopausia o se la han “curado”?"
    ],

    "Síntomas específicos": [
        "¿Qué puedo hacer con los sofocos?",
        "¿Hay algo que funcione de verdad para el calor?",
        "¿Por qué tengo tanto calor de repente?",
        "¿Los sofocos se van o se quedan para siempre?",
        "¿Por qué algunas de mis amigas casi no tienen sofocos y yo sí?",
        "¿Qué puedo hacer con el insomnio?",
        "¿Por qué estoy más cansada que antes?",
        "¿Por qué me enfermo más que antes?",
        "¿Es normal tener más infecciones ahora?",
        "¿Por qué tengo dolores nuevos que antes no tenía?",
        "¿Por qué siento que mi sistema inmunológico está peor?",
        "¿Estos síntomas son para siempre o en algún momento aflojan?"
    ],

    "Medicación y sistema sanitario": [
        "¿Por qué me recetan tantas cosas sin explicarme bien para qué son?",
        "¿Estoy tomando demasiados medicamentos?",
        "¿La menopausia se está medicalizando demasiado?",
        "¿Qué alternativas tengo a tomar medicación para todo?",
        "¿Cuándo es realmente necesaria la terapia hormonal?",
        "¿Por qué a unas mujeres se les receta terapia hormonal y a otras no?",
        "¿La medicación para la menopausia puede aumentar el riesgo de cáncer?",
        "Si como bien, ¿de verdad tengo que tomar suplementos?",
        "¿Todas las mujeres deberían tomar suplementos o tratamiento hormonal, salvo contraindicaciones?",
        "¿Tiene límite de tiempo el tratamiento hormonal sustitutivo?",
        "¿Es mejor tratar síntomas sueltos o ir a la causa?",
        "¿Cómo puedo evitar tomar medicamentos innecesarios?",
        "¿Qué decisiones médicas dependen realmente de mí y no solo del médico?"
    ],

    "Límites de la IA": [
        "¿Puede una app decirme qué tomar?",
        "¿Hasta dónde puede ayudar una IA en mi salud?",
        "¿Cómo uso una app sin sustituir al médico de toda la vida?",
        "¿Qué tipo de recomendaciones son seguras en una app?",
        "¿Cómo combinar la información de una app con la atención médica?",
        "¿Qué puede hacer una IA y qué no?",
        "¿Cómo saber cuándo tengo que ir a un profesional de la salud?",
        "¿Puede una app orientarme sin diagnosticarme?",
        "¿Cómo evitar engancharme demasiado a una app o a la IA?",
        "¿Cómo usar la tecnología para cuidarme mejor sin hacer locuras?"
    ],

    "Emoción y estado de ánimo": [
        "¿Cómo afecta la menopausia a mi estado emocional?",
        "¿Por qué estoy más irritable?",
        "¿Por qué me afectan más las cosas que antes?",
        "¿Esto emocional es hormonal o psicológico?",
        "¿Cómo influye el estrés en la menopausia?",
        "¿Por qué siento que no soy la misma?",
        "¿Cómo puedo gestionar mejor mis emociones?",
        "¿Esto es pasajero o se queda?",
        "¿Cómo recuperar algo de estabilidad emocional?",
        "¿Qué relación hay entre menopausia y ansiedad?",
        "¿Por qué estoy tan triste si, en teoría, estaba deseando no volver a tener la regla?",
        "¿Los síntomas de la menopausia se parecen a los de mi madre o mi abuela? Me da miedo acabar igual que ellas."
    ],

    "Sexo, deseo y relaciones": [
        "¿Por qué no tengo ganas de sexo si mi pareja me sigue gustando?",
        "¿A todas las mujeres se les baja la libido en esta etapa?",
        "¿Hasta qué punto la bajada de deseo tiene que ver con el miedo al deterioro físico o a dejar de ser atractiva?",
        "¿La sequedad vaginal es para siempre o se puede mejorar?",
        "¿Hay cosas que ya no se pueden hacer en la cama en esta etapa?",
        "Si cambian las hormonas y sube la testosterona, ¿eso puede cambiar lo que me atrae o de quién me enamoro?"
    ],

    "Niebla mental, memoria y cansancio cognitivo": [
        "¿Es normal que se me olviden palabras que tengo 'en la punta de la lengua' todo el rato?",
        "¿Por qué tengo la sensación de que estoy más lenta para pensar que antes?",
        "¿Esto de la 'niebla mental' es cosa de la menopausia o es que me estoy volviendo torpe?",
        "¿Cómo sé si estos despistes son normales o si deberían preocuparme?",
        "¿Por qué me cuesta más concentrarme en cosas que antes hacía sin pensar?",
        "¿Es normal acabar el día con la cabeza agotada aunque no haya hecho 'tanto'?",
        "¿Voy a recuperar mi agilidad mental o esto se queda así?",
        "¿La menopausia puede afectar a mi memoria a largo plazo o solo a estos despistes del día a día?",
        "¿Qué puedo hacer para mejorar la concentración y no sentir que tengo la cabeza 'nublada'?",
        "¿Cómo explico a los demás que no es que sea despistada, sino que estoy notando cambios mentales reales?"
    ]
}


for topic, topic_queries in queries.items():
    print(f"\nTopic: {topic}\n")
    for q in topic_queries:
        base = answer_base(q, max_new_tokens=220)
        rag_result = answer_rag(
            q,
            k=5,
            initial_k=15,
            max_per_doc=1,
            max_new_tokens=220
        )

        print("\n" + "=" * 80)
        print("QUESTION:", q)
        print("\nQUESTION TYPE:\n", rag_result["question_type"])
        if "question_intent" in rag_result:
            print("\nQUESTION INTENT:\n", rag_result["question_intent"])
        print("\nTOP SCORE:\n", rag_result["top_score"])
        if "retrieval_query" in rag_result:
            print("\nRETRIEVAL QUERY:\n", rag_result["retrieval_query"])
        if "retrieval_mode" in rag_result:
            print("\nRETRIEVAL MODE:\n", rag_result["retrieval_mode"])

        print("\nBASELINE ANSWER:")
        print(base)

        print("\nRAG ANSWER:")
        print(rag_result["answer"])

        print("\nTOP RAG SOURCES:")
        for c in rag_result["contexts"][:3]:
            print(f"- PMID={c['pmid']} | type={c['type']} | title={c['title']}")



Topic: Incertidumbre diagnóstica


QUESTION: ¿Cómo sé si lo que me está pasando es menopausia o no?

QUESTION TYPE:
 clinical_uncertainty

QUESTION INTENT:
 diagnosis

TOP SCORE:
 1.0818232460021973

RETRIEVAL QUERY:
 ¿Cómo sé si lo que me está pasando es menopausia o no? menopause diagnosis clinical diagnosis menopausal transition perimenopause 12 months amenorrhea amenorrhoea final menstrual period menstrual irregularity irregular periods vasomotor symptoms hot flashes hot flushes night sweats vaginal dryness genitourinary symptoms

RETRIEVAL MODE:
 normal

BASELINE ANSWER:
La menopausia puede ser un proceso complejo y variado en cada mujer. Algunos signos comunes de la menopausia incluyen cambios en el ciclo menstrual (como períodos irregulares o interrupción del flujo menstrual), calambres, sudoración nocturna, cambios de humor, pérdida de cabello, y disminución de la libido. Si experimentas varios de estos síntomas durante más de un año después de tu último período menstrual, es 